# Pricing Optimisation — Acceptance Probability & Expected Financial Loss

## Purpose
This notebook implements the **black-box model generation pipeline** used downstream by the pricing optimisation module.  
It produces four output CSV files consumed by the optimiser:

| Output file | Content | Model source |
|---|---|---|
| `df_acceptance_xgb_black_box.csv` | Acceptance probability per policy | XGBoost classifier |
| `df_exp_financial_loss_xgb_black_box.csv` | Expected financial loss per policy | XGBoost regressor |
| `df_acceptance_linear_model_black_box.csv` | Acceptance probability per policy | GLM (Logistic Regression) |
| `df_acceptance_gam_model_black_box.csv` | Acceptance probability per policy | GAM (LogisticGAM) |
| `df_exp_financial_loss_linear_black_box.csv` | Expected financial loss per policy | Linear  |
| 

Additionally, it serialises all trained models to `artifacts/` as pickle files for reuse (see Section 17).

## Workflow Overview
1. **Setup** — install dependencies  
2. **Data ingestion** — load and clean the 3m-1006 parquet dataset  
3. **Feature engineering** — rename columns, compute premium change ratio `U`  
4. **In sample population  filtering **
5. **Model training (cross-validation with OOF predictions)**  
   - XGBoost regressor: predicts current policy premium `Y`  
   - XGBoost classifier: predicts churn probability  
   - GLM (logistic regression): interpretable churn alternative  
   - GAM (LogisticGAM): semi-parametric churn alternative  
   - Linear regression: interpretable premium baseline  
6. **Output generation** — derive acceptance probability and save CSVs  
7. **Model serialisation** — save all four models to `artifacts/`

## Section 2 — Data Ingestion & Preprocessing

Loads the anonymised 3-month policy dataset (`full_dataset_anonimized_processed_3m_1006.parquet`) and applies the following quality filters:

- **Missing upcoming renewal data** — rows with sentinel values for premium (`9999`), premium % change (`9`) or driver age (`100`) are dropped.
- **Duplicate churn rows** — only the first churn event per policy is retained; subsequent rows after churn are removed.
- **Mismatch policies** — policies where the *supposed* next-year premium differs from the *actual* premium by more than €40 (or the % change differs by more than 0.2) are excluded entirely to avoid noisy training signal.

The analysis is restricted to `year == 7` through the `df_sub` filter below.

In [341]:
import pandas as pd

def preprocess_3m_1006(parquet_path: str):
    """"
    #### Args
    - parquet_path: path to parquet file containing the 3m_1006 dataset

    #### Returns
    - df: preprocessed dataframe
    """
    df = pd.read_parquet(parquet_path)
    # make sure df is sorted by policy number first then year
    df = df.sort_values(['encoded_policy_number', 'year']).reset_index(drop=True)
    print(f"Original rows {len(df)}, Unique policies: {df['encoded_policy_number'].nunique()}")

    # get rid of rows with missing upcoming renewal data
    df = df[
        (df['premio_total_continuado_x_impute_GTE100_LTE1500_missing_9999'] != 9999) & # upcoming renewal premium
        (df['tx_pt_continuado_x_impute_inflation_this_renewal_6m_GTEminus1_LTE1_missing_9'] != 9) & # cur renewal % change
        (df['idade_condutor_x_impute_GTE18_LTE100_map_missing_100'] != 100) # age
    ]

    # remove churned rows that are not the first churned row of each policy
    is_first_policy_row = df["encoded_policy_number"] != df["encoded_policy_number"].shift(1)
    mask = (
        (df["is_churn"] == False) |
        (
            (df["is_churn"] == True) &
            (
                is_first_policy_row |
                (df["is_churn"].shift(1) == False)
            )
        )
    )
    df = df[mask]

    # ----- remove entire policies with mismatches (supposed increase vs actual increase) 
    # ----- over the thresholds
    # premium mismatch
    supposed = "premio_total_continuado_x_impute_GTE100_LTE1500_missing_9999"
    actual = "valor_vigor_pt_apol"

    temp = df[['encoded_policy_number', actual, supposed]].copy()
    temp['diff'] = (temp[actual] - temp[supposed].shift(1)).fillna(0).abs()

    same_policy_mask = temp['encoded_policy_number'] == temp['encoded_policy_number'].shift(1)
    temp = temp[same_policy_mask]
    mismatches = temp.groupby('encoded_policy_number')['diff'].max() > 40 # threshold

    bad_policies = set(mismatches.index[mismatches])
    
    # premium % change mismatch
    actual = "tx_pt_continuado_ant_x_impute_inflation_last_renewal_6m_GTEminus1_LTE1_missing_9"
    supposed = "tx_pt_continuado_x_impute_inflation_this_renewal_6m_GTEminus1_LTE1_missing_9"

    temp = df[['encoded_policy_number', actual, supposed]].copy()
    temp = temp[temp[actual] != 9.0]
    temp['diff'] = (temp[actual] - temp[supposed].shift(1)).fillna(0).abs()

    same_policy_mask = temp['encoded_policy_number'] == temp['encoded_policy_number'].shift(1)
    temp = temp[same_policy_mask]
    mismatches = temp.groupby('encoded_policy_number')['diff'].max() > 0.2 # threshold

    bad_policies |= set(mismatches.index[mismatches])

    # remove bad policies from df
    df = df[~df['encoded_policy_number'].isin(bad_policies)]
    print(f"Remaining rows {len(df)}, Remaining unique policies: {df['encoded_policy_number'].nunique()}")

    return df


## Section 1 — Setup & Dependencies

In [250]:
from preprocessing import *
import numpy as np
import pprint as pp
from matplotlib import pyplot as plt


In [251]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder



In [252]:
import sklearn

In [253]:
file_path = r"full_dataset_anonimized_processed_3m_1006.parquet"
df = preprocess_3m_1006(file_path)

Original rows 3370487, Unique policies: 1103634
Remaining rows 3075795, Remaining unique policies: 1034510


In [254]:
df.head()

,encoded_policy_number,idade_condutor_x_impute_GTE18_LTE100_map_missing_100,indice_bonus_malus,canal_distribuicao_h4_x_grouping_all_products,cod_estatistico_classe_risco,num_sin_tot,num_apolices_tot,ind_cliente_b_x_grouping_all_products,idade_construcao_veiculo_x_impute_GTE0_LTE50_missing_0,antiguidade_apolice_x_impute_GTE0_LTE30_missing_0,valor_vigor_pt_apol,tx_pt_continuado_ant_x_impute_inflation_last_renewal_6m_GTEminus1_LTE1_missing_9,premio_total_continuado_x_impute_GTE100_LTE1500_missing_9999,tx_pt_continuado_x_impute_inflation_this_renewal_6m_GTEminus1_LTE1_missing_9,year,is_churn
0,5,86.0,50.000,19,T4005040,0,1,0,17.0,17.0,174.4700,-0.108391,157.14,-0.111998,0,False
1,5,87.0,47.500,19,T4005040,0,1,0,18.0,18.0,157.1400,-0.111998,156.74,-0.011103,1,False
2,5,88.0,45.125,19,T4005040,0,1,0,19.0,19.0,156.7400,-0.011103,156.36,-0.009236,2,False
3,5,89.0,45.000,19,T4005040,0,1,0,20.0,20.0,155.9369,-0.011918,164.11,0.051139,3,False
4,5,90.0,45.000,19,T4005040,0,1,0,21.0,21.0,164.1100,0.048342,163.72,-0.007067,4,False


In [255]:
!pip install -U glum

Defaulting to user installation because normal site-packages is not writeable


In [256]:
df_sub = df.loc[df.year==7]


## Section 3 — Feature Name Mapping

Raw column names in the parquet file use verbose Portuguese/encoded names.  
This section defines a mapping to clean `X_` prefixed names used throughout the modelling pipeline.

| Clean name | Meaning |
|---|---|
| `X_age` | Driver age (clamped 18–100) |
| `X_vehicle_age` | Vehicle age in years |
| `X_policy_tenure` | Policy seniority in years |
| `X_policy_count` | Number of policies held by this client |
| `X_risk_code` | Risk segment code (higher = lower risk) |
| `X_distr_channel` | Distribution channel group |
| `X_ttm_claims` | Total claims in trailing 12 months |
| `X_bonus_malus_rating` | Bonus-malus coefficient |
| `X_vehicle_type` | Vehicle statistical risk class |
| `X_policy_premium` | Current active premium (`Y` in regression) |
| `X_upcoming_premium` | Proposed renewal premium |
| `X_cur_renewal_perc` | % premium change at this renewal |
| `U` | Relative premium increase: `(upcoming - current) / current` |

In [257]:

age = 'idade_condutor_x_impute_GTE18_LTE100_map_missing_100'
vehicle_age = 'idade_construcao_veiculo_x_impute_GTE0_LTE50_missing_0' # 0 = new car
policy_tenure = 'antiguidade_apolice_x_impute_GTE0_LTE30_missing_0'
policy_count = 'num_apolices_tot'
risk_code = 'ind_cliente_b_x_grouping_all_products' # integers, larger = higher survival chance
distr_channel = 'canal_distribuicao_h4_x_grouping_all_products' # larger = higher obs survival in training
ttm_claims = 'num_sin_tot'
bonus_malus_rating = 'indice_bonus_malus'
vehicle_type = 'cod_estatistico_classe_risco'
policy_premium = 'valor_vigor_pt_apol'
upcoming_premium = 'premio_total_continuado_x_impute_GTE100_LTE1500_missing_9999'
prev_renewal_perc = "tx_pt_continuado_ant_x_impute_inflation_last_renewal_6m_GTEminus1_LTE1_missing_9"
cur_renewal_perc = 'tx_pt_continuado_x_impute_inflation_this_renewal_6m_GTEminus1_LTE1_missing_9' # % change of total premium
id_number = 'encoded_policy_number'

# for selecting the features to build the model
covariate_cols = [age, vehicle_age, policy_count, risk_code, distr_channel, ttm_claims,bonus_malus_rating, vehicle_type,
                  policy_premium, upcoming_premium, cur_renewal_perc]
col_name_map = {
    id_number: 'id',
    age: 'X_age',
    vehicle_age: 'X_vehicle_age',
    policy_tenure: 'X_policy_tenure',
    policy_count: 'X_policy_count',
    risk_code: 'X_risk_code',
    distr_channel: 'X_distr_channel',
    ttm_claims: 'X_ttm_claims',
    bonus_malus_rating: 'X_bonus_malus_rating',
    vehicle_type: 'X_vehicle_type',
    policy_premium: 'X_policy_premium',
    upcoming_premium: 'X_upcoming_premium',
    prev_renewal_perc:"X_prev_renewal_perc",
    cur_renewal_perc: 'X_cur_renewal_perc'
}

In [258]:
df_sub = df_sub.rename(columns=col_name_map) 

In [259]:
df_sub.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='str')

In [260]:
# common function to filter out rows fro xgb regression and classification 

## Section 4 — Outlier Filtering: `filter_middle_advanced`

`filter_middle_advanced` applies **per-column** outlier filtering before model training.  
Three methods are supported:

- **`iqr`** — keeps rows within `[Q1 − k·IQR, Q3 + k·IQR]` (default `k = 1.5`)
- **`percentile`** — keeps rows between specified quantiles (e.g., 5th–95th)
- **`zscore`** — keeps rows within `mean ± threshold·std`

Categorical filtering reduces cardinality by keeping the top-N most frequent values.

**Why filter?**  
Extreme premium values and very rare distribution channels create poor generalisation.  
The filtered datasets vary per model (XGBoost, GLM, GAM, Linear) to allow independent tuning of cleaning thresholds.

In [261]:
def filter_middle_advanced(df, 
                          numeric_filters=None, 
                          categorical_filters=None,
                          keep_id_cols=None):
    """
    Advanced filtering with per-column configuration
    
    Parameters:
    -----------
    numeric_filters : dict
        {column_name: {'multiplier': 1.5, 'method': 'iqr'}}
    categorical_filters : dict
        {column_name: {'min_freq': 0.01, 'max_categories': 10, 'keep_values': [...]}}
    keep_id_cols : list
        Columns to preserve in output (like 'encoded_policy_number')
    """
    df_filtered = df.copy()
    
    if keep_id_cols is None:
        keep_id_cols = ['encoded_policy_number']
    
    # Numeric filtering
    if numeric_filters:
        for col, config in numeric_filters.items():
            multiplier = config.get('multiplier', 1.5)
            method = config.get('method', 'iqr')
            
            if method == 'iqr':
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower = Q1 - multiplier * IQR
                upper = Q3 + multiplier * IQR
            elif method == 'percentile':
                lower = df[col].quantile(config.get('lower_q', 0.05))
                upper = df[col].quantile(config.get('upper_q', 0.95))
            elif method == 'zscore':
                mean = df[col].mean()
                std = df[col].std()
                threshold = config.get('threshold', 3)
                lower = mean - threshold * std
                upper = mean + threshold * std
            
            df_filtered = df_filtered[(df_filtered[col] >= lower) & (df_filtered[col] <= upper)]
            print(f"{col}: [{lower:.2f}, {upper:.2f}]")
    
    # Categorical filtering
    if categorical_filters:
        for col, config in categorical_filters.items():
            # Option 1: Explicit list of values to keep
            if 'keep_values' in config:
                valid_values = config['keep_values']
            # Option 2: By frequency or top N
            else:
                value_counts = df[col].value_counts()
                
                if 'max_categories' in config:
                    valid_values = value_counts.nlargest(config['max_categories']).index.tolist()
                elif 'min_freq' in config:
                    min_count = int(len(df) * config['min_freq'])
                    valid_values = value_counts[value_counts >= min_count].index.tolist()
                else:
                    valid_values = value_counts.index.tolist()
            
            df_filtered = df_filtered[df_filtered[col].isin(valid_values)]
            print(f"{col}: Kept {len(valid_values)} categories")
    
    print(f"\nTotal: {len(df)} → {len(df_filtered)} rows")
    
    return df_filtered


# Usage
df_middle = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


## Section 5 — Model Architecture Definitions

Three model classes are defined here and reused across datasets:

### `ModelRegressor`
XGBoost regressor with **Optuna hyperparameter tuning**.  
- Objective: `reg:absoluteerror` (MAE-optimised)
- Tuned parameters: `learning_rate`, `max_depth`, `subsample`, `colsample_bytree`, `min_child_weight`
- Used to predict `X_policy_premium` (target `Y`)

### `ModelClassifier`
XGBoost binary classifier with **Optuna hyperparameter tuning**.  
- Metric: ROC-AUC
- Used to predict churn (`is_churn`), which is then converted to acceptance probability: `Z = 1 − is_churn`, `prob_acceptance = 1 − churn_proba`

### `ModelRegressorCV`  
Extended version of `ModelRegressor` that runs **fully nested Optuna + KFold CV** in a single call. Used for the policy premium regression CV experiment.

In [262]:

from sklearn import metrics
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from optuna.trial import Trial
N_ESTIMATORS = 200
OBJECTIVE = 'reg:absoluteerror'
class ModelRegressor():
    
    def objective(self,trial : Trial, X_train : pd.DataFrame, y_train : pd.Series,X_val : pd.DataFrame, y_val : pd.Series)-> float:
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.05, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        }
        model = xgb.XGBRegressor(**params,random_state=123,n_estimators=N_ESTIMATORS,early_stopping_rounds=5,objective = OBJECTIVE)
        model.fit(X_train, y_train,verbose=0,eval_set=[(X_val, y_val)])
        y_pred = model.predict(X_val)
        loss = metrics.mean_absolute_error(y_true=y_val,y_pred=y_pred)
        return loss
    
    def tuning(self,X_train:pd.DataFrame,y_train:pd.Series,X_val:pd.DataFrame, y_val:pd.Series,n_trials:int)->dict:
        y_train = y_train
        y_val = y_val
        study = optuna.create_study(study_name='Xgboost', direction='minimize')
        study.optimize(lambda trial: self.objective(trial, X_train, y_train, X_val, y_val), n_trials=n_trials)
        best_params = study.best_params
        return best_params
    
    def train(self,X_train:pd.DataFrame,y_train:pd.Series,X_val:pd.DataFrame, y_val:pd.Series,best_params:dict)->xgb.XGBRegressor:
        y_train = y_train
        y_val = y_val
        model = xgb.XGBRegressor(**best_params,random_state=123,n_estimators=N_ESTIMATORS,early_stopping_rounds=5,objective = OBJECTIVE)
        model.fit(X_train, y_train,verbose=0,eval_set=[(X_val, y_val)])
        return model

In [263]:
from sklearn import metrics
import pandas as pd
import xgboost as xgb
import optuna
from optuna.trial import Trial

N_ESTIMATORS = 200

class ModelClassifier():
    
    def objective(self,trial : Trial, X_train : pd.DataFrame, y_train : pd.Series,X_val : pd.DataFrame, y_val : pd.Series)-> float:
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.05, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        }

        model = xgb.XGBClassifier(**params,random_state=123,n_estimators=N_ESTIMATORS,early_stopping_rounds=5)
        model.fit(X_train, y_train,verbose=0,eval_set=[(X_val, y_val)])
        y_score = model.predict_proba(X_val)[:,1]
        roc_auc = metrics.roc_auc_score(y_true=y_val,y_score=y_score)
        return roc_auc
    
    def tuning(self,X_train:pd.DataFrame,y_train:pd.Series,X_val:pd.DataFrame, y_val:pd.Series,n_trials:int)->dict:
        study = optuna.create_study(study_name='Xgboost', direction='maximize')
        study.optimize(lambda trial: self.objective(trial, X_train, y_train, X_val, y_val), n_trials=n_trials)
        best_params = study.best_params
        return best_params
    
    def train(self,X_train:pd.DataFrame,y_train:pd.Series,X_val:pd.DataFrame, y_val:pd.Series,best_params:dict)->xgb.XGBClassifier:
        model = xgb.XGBClassifier(**best_params,random_state= 123,n_estimators= N_ESTIMATORS,
                                  early_stopping_rounds= 5)
        model.fit(X_train, y_train,verbose=0,eval_set=[(X_val, y_val)])
        return model


In [264]:
import numpy as np
import pandas as pd


class FeatureProcessor:
    """
    Center+sphere numeric features and label-encode categorical features as continuous values.
    Supports PCA with whitening for multivariate Mahalanobis distance.
    """

    def __init__(
        self,
        numeric_cols=None,
        categorical_cols=None,
        regularization=1e-6,
        missing_category="__MISSING__",
        use_pca=False,
        n_components=None,
        explained_variance_threshold=None,
    ):
        """
        Parameters
        ----------
        numeric_cols : list, optional
            Columns to treat as numeric. If None, inferred from data.
        categorical_cols : list, optional
            Columns to treat as categorical. If None, inferred from data.
        regularization : float, default=1e-6
            Regularization added to eigenvalues for numerical stability.
        missing_category : str, default="__MISSING__"
            Value to use for missing categorical values.
        use_pca : bool, default=False
            If True, apply PCA with whitening to numeric features.
            Whitening transforms data so Euclidean distance = Mahalanobis distance.
        n_components : int or float, optional
            Number of PCA components to keep:
            - If int: keep exactly n_components
            - If float (0.0, 1.0): keep components explaining this fraction of variance
            - If None: keep all components (full rank with whitening)
        explained_variance_threshold : float, optional
            Alternative to n_components: keep components until cumulative
            explained variance reaches this threshold (e.g., 0.95 for 95%).
        """
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols
        self.regularization = regularization
        self.missing_category = missing_category
        self.use_pca = use_pca
        self.n_components = n_components
        self.explained_variance_threshold = explained_variance_threshold

    def fit(self, X: pd.DataFrame):
        X = X.copy()

        if self.numeric_cols is None:
            self.numeric_cols_ = X.select_dtypes(include=[np.number]).columns.tolist()
        else:
            self.numeric_cols_ = list(self.numeric_cols)

        if self.categorical_cols is None:
            self.categorical_cols_ = [c for c in X.columns if c not in self.numeric_cols_]
        else:
            self.categorical_cols_ = list(self.categorical_cols)

        # Numeric centering + sphering/PCA
        if self.numeric_cols_:
            X_num = X[self.numeric_cols_].astype(float)
            self.numeric_means_ = X_num.mean(axis=0)
            X_num_centered = X_num - self.numeric_means_

            # Compute covariance matrix
            cov = np.cov(X_num_centered.values, rowvar=False)
            if np.ndim(cov) == 0:
                cov = np.array([[float(cov)]])

            # Eigendecomposition
            eigvals, eigvecs = np.linalg.eigh(cov)
            
            # Sort eigenvalues and eigenvectors in descending order
            idx = np.argsort(eigvals)[::-1]
            eigvals = eigvals[idx]
            eigvecs = eigvecs[:, idx]
            
            # Regularize eigenvalues
            eigvals = np.maximum(eigvals, self.regularization)
            
            # Store full eigenvalues for explained variance
            self.eigenvalues_ = eigvals
            total_var = eigvals.sum()
            self.explained_variance_ratio_ = eigvals / total_var if total_var > 0 else eigvals
            self.cumulative_variance_ratio_ = np.cumsum(self.explained_variance_ratio_)

            if self.use_pca:
                # Determine number of components to keep
                n_features = len(eigvals)
                if self.explained_variance_threshold is not None:
                    # Keep components until threshold is reached
                    n_keep = np.searchsorted(self.cumulative_variance_ratio_, 
                                            self.explained_variance_threshold) + 1
                    n_keep = min(n_keep, n_features)
                elif self.n_components is not None:
                    if isinstance(self.n_components, float) and 0 < self.n_components < 1:
                        # n_components as fraction of variance
                        n_keep = np.searchsorted(self.cumulative_variance_ratio_, 
                                                self.n_components) + 1
                        n_keep = min(n_keep, n_features)
                    else:
                        # n_components as integer
                        n_keep = min(int(self.n_components), n_features)
                else:
                    # Keep all components
                    n_keep = n_features
                
                self.n_components_ = n_keep
                
                # Select top components
                eigvals_keep = eigvals[:n_keep]
                eigvecs_keep = eigvecs[:, :n_keep]
                
                # PCA with whitening: transform = X_centered @ eigvecs @ diag(1/sqrt(eigvals))
                # This makes covariance matrix of transformed data = Identity
                self.pca_components_ = eigvecs_keep
                self.whitening_scale_ = 1.0 / np.sqrt(eigvals_keep)
                
                # Combined transformation matrix for efficiency
                self.sphering_matrix_ = eigvecs_keep @ np.diag(self.whitening_scale_)
                
                # Feature names for PCA components
                self.numeric_feature_names_ = [f"PC{i+1}" for i in range(n_keep)]
            else:
                # Original sphering (full rank)
                self.n_components_ = len(eigvals)
                self.sphering_matrix_ = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T
                self.numeric_feature_names_ = [f"sphered__{col}" for col in self.numeric_cols_]
        else:
            self.numeric_means_ = pd.Series(dtype=float)
            self.sphering_matrix_ = np.empty((0, 0))
            self.n_components_ = 0
            self.numeric_feature_names_ = []

        # Categorical label encoding -> continuous [0, 1]
        self.cat_mapping_ = {}
        self.cat_denominator_ = {}
        self.encoded_cat_feature_names_ = []

        for col in self.categorical_cols_:
            col_values = X[col].fillna(self.missing_category).astype(str)
            categories = pd.Index(col_values.unique())
            mapping = {cat: idx for idx, cat in enumerate(categories)}
            self.cat_mapping_[col] = mapping
            # Keep one extra bucket for unseen categories at transform time.
            self.cat_denominator_[col] = max(len(mapping), 1)
            self.encoded_cat_feature_names_.append(f"label_cont__{col}")

        self.output_feature_names_ = [
            *self.numeric_feature_names_,
            *self.encoded_cat_feature_names_,
        ]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()

        # Transform numeric features
        if self.numeric_cols_:
            X_num = X[self.numeric_cols_].astype(float)
            X_num_centered = X_num - self.numeric_means_
            X_num_transformed = X_num_centered.values @ self.sphering_matrix_
        else:
            X_num_transformed = np.empty((len(X), 0))

        # Transform categorical features using fitted mappings
        cat_arrays = []
        for col in self.categorical_cols_:
            values = X[col].fillna(self.missing_category).astype(str)
            mapping = self.cat_mapping_[col]
            unknown_code = len(mapping)
            denom = self.cat_denominator_[col]

            label_codes = values.map(mapping).fillna(unknown_code).astype(float)
            continuous_codes = (label_codes / denom).to_numpy().reshape(-1, 1)
            cat_arrays.append(continuous_codes)

        if cat_arrays:
            X_cat_encoded = np.hstack(cat_arrays)
        else:
            X_cat_encoded = np.empty((len(X), 0))

        X_out = np.hstack([X_num_transformed, X_cat_encoded])
        return pd.DataFrame(X_out, index=X.index, columns=self.output_feature_names_)

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)
    
    def get_explained_variance_info(self):
        """
        Get information about explained variance (useful when use_pca=True).
        
        Returns
        -------
        dict with keys:
            - 'eigenvalues': array of eigenvalues
            - 'explained_variance_ratio': variance explained by each component
            - 'cumulative_variance_ratio': cumulative variance explained
            - 'n_components_kept': number of components retained
        """
        if not hasattr(self, 'eigenvalues_'):
            raise ValueError("Model not fitted yet. Call fit() first.")
        
        return {
            'eigenvalues': self.eigenvalues_,
            'explained_variance_ratio': self.explained_variance_ratio_,
            'cumulative_variance_ratio': self.cumulative_variance_ratio_,
            'n_components_kept': self.n_components_,
        }
    
    def inverse_transform_numeric(self, X_transformed: np.ndarray) -> np.ndarray:
        """
        Inverse transform for numeric features only (approximate if PCA was used with n_components < n_features).
        
        Parameters
        ----------
        X_transformed : ndarray of shape (n_samples, n_components)
            Transformed numeric features only (not including categorical).
            
        Returns
        -------
        X_original : ndarray of shape (n_samples, n_numeric_features)
            Approximate reconstruction of original numeric features.
        """
        if not self.use_pca:
            # For full-rank sphering, inverse is more complex
            # X_sphered = X_centered @ sphering_matrix
            # X_centered = X_sphered @ sphering_matrix^-1
            raise NotImplementedError("Inverse transform only supported when use_pca=True")
        
        if not self.numeric_cols_:
            return np.empty((len(X_transformed), 0))
        
        # Inverse: X_centered = PC_scores @ components.T
        X_centered = X_transformed @ self.pca_components_.T
        
        # Add back the mean
        X_original = X_centered + self.numeric_means_.values
        
        return X_original

In [265]:
# Demo: FeatureProcessor with PCA and whitening

# Example 1: Standard sphering (original behavior)
fp_standard = FeatureProcessor()
X_sphered = fp_standard.fit_transform(X_raw)
print(f"Standard sphering - Output shape: {X_sphered.shape}")

# Example 2: PCA with whitening, keeping top 5 components
fp_pca = FeatureProcessor(use_pca=True, n_components=5)
X_pca = fp_pca.fit_transform(X_raw)
print(f"\nPCA (5 components) - Output shape: {X_pca.shape}")
var_info = fp_pca.get_explained_variance_info()
print(f"Explained variance by top 5: {var_info['cumulative_variance_ratio'][:5][-1]:.3f}")

# Example 3: PCA keeping 95% of variance
fp_pca_var = FeatureProcessor(use_pca=True, explained_variance_threshold=0.95)
X_pca_var = fp_pca_var.fit_transform(X_raw)
print(f"\nPCA (95% variance) - Output shape: {X_pca_var.shape}")
print(f"Components kept: {fp_pca_var.n_components_}")

# Example 4: PCA with all components (whitening only, no dimensionality reduction)
fp_pca_full = FeatureProcessor(use_pca=True)
X_pca_full = fp_pca_full.fit_transform(X_raw)
print(f"\nPCA (all components) - Output shape: {X_pca_full.shape}")

# Note: With whitening, Euclidean distance in transformed space = Mahalanobis distance in original space
# This is useful for distance-based algorithms like k-NN, k-means, etc.

Standard sphering - Output shape: (194373, 10)

PCA (5 components) - Output shape: (194373, 5)
Explained variance by top 5: 0.996

PCA (95% variance) - Output shape: (194373, 2)
Components kept: 2

PCA (all components) - Output shape: (194373, 10)


In [266]:
df_middle_class = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encode_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [267]:
def filter_middle_iqr(df, columns=None, multiplier=1.5):
    """Keep observations within Q1 - multiplier*IQR to Q3 + multiplier*IQR"""
    df_filtered = df.copy()
    
    # Use numeric columns only if not specified
    if columns is None:
        columns = df.select_dtypes(include=['number']).columns
        columns = [col for col in columns if col not in ['encoded_policy_number', 'is_churn']]
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR
        df_filtered = df_filtered[(df_filtered[col] >= lower) & (df_filtered[col] <= upper)]
    
    return df_filtered



In [268]:
df_middle_class.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='str')

In [269]:
from sklearn import metrics
from sklearn.model_selection import KFold, cross_val_score
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from optuna.trial import Trial

N_ESTIMATORS = 200
OBJECTIVE = 'reg:absoluteerror'
N_FOLDS = 5
RANDOM_STATE = 123

class ModelRegressorCV():
    
    def __init__(self, n_folds=N_FOLDS, random_state=RANDOM_STATE):
        self.n_folds = n_folds
        self.random_state = random_state
        self.best_params = None
        self.cv_scores = None
        self.final_model = None
        
    def objective(self, trial: Trial, X: pd.DataFrame, y: pd.Series) -> float:
        """Optuna objective with cross-validation"""
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.05, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        }
        
        # Cross-validation with the suggested parameters
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        cv_scores = []
        
        for train_idx, val_idx in kfold.split(X):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Apply log transformation
            y_train_transformed = np.log(y_train_fold) + 1
            y_val_transformed = np.log(y_val_fold) + 1
            
            model = xgb.XGBRegressor(
                **params,
                random_state=self.random_state,
                n_estimators=N_ESTIMATORS,
                early_stopping_rounds=5,
                objective=OBJECTIVE
            )
            
            model.fit(
                X_train_fold, 
                y_train_transformed,
                verbose=0,
                eval_set=[(X_val_fold, y_val_transformed)]
            )
            
            y_pred = model.predict(X_val_fold)
            fold_mae = metrics.mean_absolute_error(y_val_transformed, y_pred)
            cv_scores.append(fold_mae)
        
        # Return mean CV score
        return np.mean(cv_scores)
    
    def tuning(self, X: pd.DataFrame, y: pd.Series, n_trials: int) -> dict:
        """Hyperparameter tuning with cross-validation"""
        print(f"Starting hyperparameter tuning with {n_trials} trials and {self.n_folds}-fold CV...")
        
        study = optuna.create_study(study_name='XGBoost_CV', direction='minimize')
        study.optimize(
            lambda trial: self.objective(trial, X, y), 
            n_trials=n_trials,
            show_progress_bar=True
        )
        
        self.best_params = study.best_params
        print(f"\nBest CV MAE: {study.best_value:.6f}")
        print(f"Best parameters: {self.best_params}")
        
        return self.best_params
    
    def train_with_cv(self, X: pd.DataFrame, y: pd.Series, params: dict = None) -> tuple:
        """Train model with cross-validation and return CV scores"""
        if params is None:
            if self.best_params is None:
                raise ValueError("No parameters provided. Run tuning() first or provide params.")
            params = self.best_params
        
        print(f"\nTraining with {self.n_folds}-fold cross-validation...")
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        
        cv_scores = []
        fold_models = []
        
        for fold_num, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Apply log transformation
            y_train_transformed = y_train_fold 
            y_val_transformed = y_val_fold 
            
            model = xgb.XGBRegressor(
                **params,
                random_state=self.random_state,
                n_estimators=N_ESTIMATORS,
                early_stopping_rounds=5,
                objective=OBJECTIVE
            )
            
            model.fit(
                X_train_fold, 
                y_train_transformed,
                verbose=0,
                eval_set=[(X_val_fold, y_val_transformed)]
            )
            
            y_pred = model.predict(X_val_fold)
            fold_mae = metrics.mean_absolute_error(y_val_transformed, y_pred)
            cv_scores.append(fold_mae)
            fold_models.append(model)
            
            print(f"Fold {fold_num}: MAE = {fold_mae:.6f}")
        
        self.cv_scores = cv_scores
        print(f"\nMean CV MAE: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")
        
        return fold_models, cv_scores
    
    def train_final(self, X: pd.DataFrame, y: pd.Series, params: dict = None) -> xgb.XGBRegressor:
        """Train final model on full dataset"""
        if params is None:
            if self.best_params is None:
                raise ValueError("No parameters provided. Run tuning() first or provide params.")
            params = self.best_params
        
        print("\nTraining final model on full dataset...")
        y_transformed = np.log(y) + 1
        
        # Use a portion for validation (20%)
        split_idx = int(len(X) * 0.8)
        X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_val = y_transformed.iloc[:split_idx], y_transformed.iloc[split_idx:]
        
        model = xgb.XGBRegressor(
            **params,
            random_state=self.random_state,
            n_estimators=N_ESTIMATORS,
            early_stopping_rounds=5,
            objective=OBJECTIVE
        )
        
        model.fit(
            X_train, 
            y_train,
            verbose=0,
            eval_set=[(X_val, y_val)]
        )
        
        self.final_model = model
        print("Final model training complete!")
        
        return model
    
    def predict(self, X: pd.DataFrame, inverse_transform: bool = True) -> np.ndarray:
        """Make predictions using the final model"""
        if self.final_model is None:
            raise ValueError("No model trained. Run train_final() first.")
        
        predictions = self.final_model.predict(X)
        
        if inverse_transform:
            # Inverse log transformation
            predictions = predictions
        
        return predictions


In [270]:
from sklearn.model_selection import KFold, cross_val_score
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from optuna.trial import Trial

N_ESTIMATORS = 200
OBJECTIVE = 'reg:absoluteerror'
N_FOLDS = 5
RANDOM_STATE = 123

class ModelRegressorCV():
    
    def __init__(self, n_folds=N_FOLDS, random_state=RANDOM_STATE):
        self.n_folds = n_folds
        self.random_state = random_state
        self.best_params = None
        self.cv_scores = None
        self.final_model = None
        
    def objective(self, trial: Trial, X: pd.DataFrame, y: pd.Series) -> float:
        """Optuna objective with cross-validation"""
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.05, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        }
        
        # Cross-validation with the suggested parameters
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        cv_scores = []
        
        for train_idx, val_idx in kfold.split(X):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Apply log transformation
            y_train_transformed = np.log(y_train_fold) + 1
            y_val_transformed = np.log(y_val_fold) + 1
            
            model = xgb.XGBRegressor(
                **params,
                random_state=self.random_state,
                n_estimators=N_ESTIMATORS,
                early_stopping_rounds=5,
                objective=OBJECTIVE
            )
            
            model.fit(
                X_train_fold, 
                y_train_transformed,
                verbose=0,
                eval_set=[(X_val_fold, y_val_transformed)]
            )
            
            y_pred = model.predict(X_val_fold)
            fold_mae = metrics.mean_absolute_error(y_val_transformed, y_pred)
            cv_scores.append(fold_mae)
        
        # Return mean CV score
        return np.mean(cv_scores)
    
    def tuning(self, X: pd.DataFrame, y: pd.Series, n_trials: int) -> dict:
        """Hyperparameter tuning with cross-validation"""
        print(f"Starting hyperparameter tuning with {n_trials} trials and {self.n_folds}-fold CV...")
        
        study = optuna.create_study(study_name='XGBoost_CV', direction='minimize')
        study.optimize(
            lambda trial: self.objective(trial, X, y), 
            n_trials=n_trials,
            show_progress_bar=True
        )
        
        self.best_params = study.best_params
        print(f"\nBest CV MAE: {study.best_value:.6f}")
        print(f"Best parameters: {self.best_params}")
        
        return self.best_params
    
    def train_with_cv(self, X: pd.DataFrame, y: pd.Series, params: dict = None) -> tuple:
        """Train model with cross-validation and return CV scores"""
        if params is None:
            if self.best_params is None:
                raise ValueError("No parameters provided. Run tuning() first or provide params.")
            params = self.best_params
        
        print(f"\nTraining with {self.n_folds}-fold cross-validation...")
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        
        cv_scores = []
        fold_models = []
        
        for fold_num, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Apply log transformation
            y_train_transformed = np.log(y_train_fold) + 1
            y_val_transformed = np.log(y_val_fold) + 1
            
            model = xgb.XGBRegressor(
                **params,
                random_state=self.random_state,
                n_estimators=N_ESTIMATORS,
                early_stopping_rounds=5,
                objective=OBJECTIVE
            )
            
            model.fit(
                X_train_fold, 
                y_train_transformed,
                verbose=0,
                eval_set=[(X_val_fold, y_val_transformed)]
            )
            
            y_pred = model.predict(X_val_fold)
            fold_mae = metrics.mean_absolute_error(y_val_transformed, y_pred)
            cv_scores.append(fold_mae)
            fold_models.append(model)
            
            print(f"Fold {fold_num}: MAE = {fold_mae:.6f}")
        
        self.cv_scores = cv_scores
        print(f"\nMean CV MAE: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")
        
        return fold_models, cv_scores
    
    def train_final(self, X: pd.DataFrame, y: pd.Series, params: dict = None) -> xgb.XGBRegressor:
        """Train final model on full dataset"""
        if params is None:
            if self.best_params is None:
                raise ValueError("No parameters provided. Run tuning() first or provide params.")
            params = self.best_params
        
        print("\nTraining final model on full dataset...")
        y_transformed = np.log(y) + 1
        
        # Use a portion for validation (20%)
        split_idx = int(len(X) * 0.8)
        X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_val = y_transformed.iloc[:split_idx], y_transformed.iloc[split_idx:]
        
        model = xgb.XGBRegressor(
            **params,
            random_state=self.random_state,
            n_estimators=N_ESTIMATORS,
            early_stopping_rounds=5,
            objective=OBJECTIVE
        )
        
        model.fit(
            X_train, 
            y_train,
            verbose=0,
            eval_set=[(X_val, y_val)]
        )
        
        self.final_model = model
        print("Final model training complete!")
        
        return model
    
    def predict(self, X: pd.DataFrame, inverse_transform: bool = True) -> np.ndarray:
        """Make predictions using the final model"""
        if self.final_model is None:
            raise ValueError("No model trained. Run train_final() first.")
        
        predictions = self.final_model.predict(X)
        
        if inverse_transform:
            # Inverse log transformation
            predictions = np.exp(predictions - 1)
        
        return predictions

In [271]:
# encode category feats

In [272]:
model_features_class = ['X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure','X_policy_premium','U']

model_features_reg = ['X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure']

In [273]:
cat_cols = list(df_middle.select_dtypes('object').columns)

from sklearn.preprocessing import LabelEncoder
for cat in cat_cols:
    le = LabelEncoder()
    le.fit(df_middle[cat])
    df_middle[cat] = le.transform(df_middle[cat])
    df_middle_class[cat] = le.transform(df_middle_class[cat])


C:\Users\malosett\AppData\Local\Temp\ipykernel_13900\2776194888.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = list(df_middle.select_dtypes('object').columns)


## Section 6 — Dataset Preparation for XGBoost

Two filtered datasets are created from `df_sub`:

- **`df_middle`** — used for the **premium regressor** (target: `X_policy_premium`)  
- **`df_middle_class`** — used for the **churn classifier** (target: `is_churn`)

Both use identical `filter_middle_advanced` thresholds.  
Categorical columns (`X_distr_channel`, `X_risk_code`, `X_vehicle_type`) are **label-encoded** here so XGBoost can consume them directly.

The premium change ratio `U` is computed as:  
$$U = \frac{\text{upcoming\_premium} - \text{policy\_premium}}{\text{policy\_premium}}$$  
This is a core feature driving churn behaviour: larger premium increases lead to higher churn probability.  
`U` is later adjusted to `1 + U` (i.e., the *uplift factor*) for the optimisation module.

In [274]:
df_middle.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='str')

In [275]:
df_middle['U'] = (df_middle['X_upcoming_premium'] - df_middle['X_policy_premium']) / df_middle['X_policy_premium']
df_middle_class['U'] = (df_middle_class['X_upcoming_premium'] - df_middle_class['X_policy_premium']) / df_middle_class['X_policy_premium']

In [276]:
# Split X and U so U stays untouched (policy-space representation)
u_cols = [
    col
    for col in ["id","U"]
    if col in df_middle_class.columns
]
U_raw = df_middle_class[u_cols].copy()

x_feature_cols = [col for col in model_features_class if col not in u_cols]
X_raw = df_middle_class[x_feature_cols].copy()

feature_processor = FeatureProcessor()
X_processed = feature_processor.fit_transform(X_raw)

print("U columns kept untouched:", u_cols)
print("U shape (untouched):", U_raw.shape)
print("X raw shape:", X_raw.shape)
print("X processed shape:", X_processed.shape)

# If needed downstream, concatenate untouched U with processed X
UX_processed = pd.concat([U_raw, X_processed], axis=1)
UX_processed.head()

U columns kept untouched: ['id', 'U']
U shape (untouched): (194373, 2)
X raw shape: (194373, 10)
X processed shape: (194373, 10)


,id,U,sphered__X_age,sphered__X_bonus_malus_rating,sphered__X_distr_channel,sphered__X_vehicle_type,sphered__X_ttm_claims,sphered__X_policy_count,sphered__X_risk_code,sphered__X_vehicle_age,sphered__X_policy_tenure,sphered__X_policy_premium
7,5,0.123099,2.298428,0.264406,-0.058562,2.621382,-0.185054,-1.160296,-0.066957,0.331389,5.691633,-0.384604
16,85,0.107047,1.401397,0.093772,-0.030788,-0.634504,-0.074525,-0.231868,-0.361906,-2.243026,5.951082,-0.377904
24,93,0.096895,1.283066,0.140256,-0.045808,-0.465812,-0.184750,-0.873439,-0.260413,0.094712,5.828775,-0.576589
32,119,0.097209,-0.679815,-0.108967,-0.019604,-0.393850,-0.271498,-0.805693,-0.255897,-0.007691,6.124242,-0.527929
118,395,0.102011,-0.667820,-0.063739,-1.088519,0.107791,-0.259477,0.729975,-0.156885,-0.507888,6.123292,-0.564618


In [277]:
model_features_class = ['X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure','X_policy_premium','U']

model_features_reg = ['X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure']

In [278]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn import metrics
import pandas as pd
import numpy as np
import xgboost as xgb

# Assuming 'df' is your dataframe with the columns you mentioned
# First, filter to middle of distribution (using IQR method)
def filter_middle_distribution(df, multiplier=1.5):
    """Remove outliers using IQR method"""
    df_filtered = df.copy()
    
    # Select numeric columns, exclude ID and target
    numeric_cols = df.select_dtypes(include=['number']).columns
    numeric_cols = [col for col in numeric_cols if col not in ['encoded_policy_number', 'is_churn']]
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR
        df_filtered = df_filtered[(df_filtered[col] >= lower) & (df_filtered[col] <= upper)]
    
    print(f"Original rows: {len(df)}, Filtered rows: {len(df_filtered)}")
    return df_filtered

# Apply filtering


# Separate features and target
X = df_middle.loc[:, model_features_reg]
y = df_middle['X_upcoming_premium']

# Initialize the model
model_regressor = ModelRegressor()

# Cross-validation approach 1: Using sklearn's cross_val_score
kfold = KFold(n_splits=5, shuffle=True, random_state=123)

# Simple CV without hyperparameter tuning
default_params = {
    'learning_rate': 0.01,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'random_state': 123,
    'n_estimators': 200,
    'objective': 'reg:absoluteerror'
}

xgb_model = xgb.XGBRegressor(**default_params, enable_categorical=True)

# Perform cross-validation
# n_jobs=1: avoids joblib spawning worker processes that may pick up a
# different numpy version than the kernel, causing BrokenProcessPool.
cv_scores = cross_val_score(
    xgb_model, 
    X, 
    y,
    cv=kfold, 
    scoring='neg_mean_absolute_error',
    n_jobs=1
)

print(f"CV MAE Scores: {-cv_scores}")
print(f"Mean CV MAE: {-cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV MAE Scores: [35.92597784 35.24692809 35.43542505 35.24695861 35.34396943]
Mean CV MAE: 35.4399 (+/- 0.2530)


In [279]:
df_middle.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'U'],
      dtype='str')

## Section 7 — XGBoost Policy Premium Regression (5-Fold OOF)

**Target variable:** `X_policy_premium` (current active premium, `Y`)  
**Features:** `model_features_reg` — 9 policy/client covariates (excluding premium-related features)

`cross_validate_with_oof_predictions` runs a **nested cross-validation**:
1. Outer KFold (5 splits) — for generating honest out-of-fold (OOF) predictions  
2. Inner 80/20 split — for Optuna hyperparameter tuning within each fold  

**Outputs stored in `df_middle`:**
- `fold_number` — which fold each row was validated in
- `oof_prediction` (`Y_hat`) — predicted policy premium  
- `oof_residual` / `oof_absolute_error` — residual diagnostics  

These OOF predictions are used as the regressor output in `df_exp_financial_loss_xgb_black_box.csv`.

In [280]:
df_middle['U'].describe()

count    194373.000000
mean          0.083095
std           0.029994
min          -0.001829
25%           0.061570
50%           0.094024
75%           0.103913
max           0.418037
Name: U, dtype: float64

In [281]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

def cross_validate_with_oof_predictions(X, y, n_splits=5, n_trials=30):
    """
    Perform cross-validation and generate out-of-fold predictions.
    FeatureProcessor is fit on each training fold and applied to both train and val
    to prevent data leakage.
    """
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)

    fold_numbers   = np.zeros(len(X))
    oof_predictions = np.zeros(len(X))

    fold_results        = []
    best_params_per_fold = []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'='*50}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'='*50}")

        fold_numbers[val_idx] = fold_idx + 1

        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        # --- fit preprocessor on train fold only (no leakage) ---
        fp = FeatureProcessor()
        X_train_proc = fp.fit_transform(X_train_fold)
        X_val_proc   = fp.transform(X_val_fold)

        from sklearn.model_selection import train_test_split
        X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
            X_train_proc, y_train_fold, test_size=0.2, random_state=123
        )

        model_reg   = ModelRegressor()
        best_params = model_reg.tuning(
            X_train_tune, y_train_tune,
            X_val_tune,   y_val_tune,
            n_trials=n_trials
        )

        print(f"Best params for fold {fold_idx + 1}: {best_params}")
        best_params_per_fold.append(best_params)

        final_model = model_reg.train(
            X_train_proc, y_train_fold,
            X_val_proc,   y_val_fold,
            best_params
        )

        y_pred_log = final_model.predict(X_val_proc)
        oof_predictions[val_idx] = y_pred_log

        mae_log      = metrics.mean_absolute_error(y_val_fold, y_pred_log)
        mae_original = metrics.mean_absolute_error(y_val_fold, oof_predictions[val_idx])

        fold_results.append({
            'fold':               fold_idx + 1,
            'mae_log_scale':      mae_log,
            'mae_original_scale': mae_original,
            'n_train':            len(train_idx),
            'n_val':              len(val_idx)
        })

        print(f"Fold {fold_idx + 1} MAE (log scale):      {mae_log:.4f}")
        print(f"Fold {fold_idx + 1} MAE (original scale): {mae_original:.4f}")

    results_df = pd.DataFrame(fold_results)

    print(f"\n{'='*50}")
    print("Cross-Validation Summary")
    print(f"{'='*50}")
    print(f"Mean MAE (log scale):      {results_df['mae_log_scale'].mean():.4f} (+/- {results_df['mae_log_scale'].std():.4f})")
    print(f"Mean MAE (original scale): {results_df['mae_original_scale'].mean():.4f} (+/- {results_df['mae_original_scale'].std():.4f})")

    overall_oof_mae = metrics.mean_absolute_error(y, oof_predictions)
    print(f"\nOverall OOF MAE: {overall_oof_mae:.4f}")

    return fold_numbers, oof_predictions, results_df, best_params_per_fold


X = df_middle.loc[:, model_features_reg]
y = df_middle['X_policy_premium']

fold_numbers, oof_predictions, results_df, best_params_list = cross_validate_with_oof_predictions(
    X, y, n_splits=5, n_trials=30
)

df_middle['fold_number']    = fold_numbers.astype(int)
df_middle['oof_prediction'] = oof_predictions
df_middle['oof_residual']       = df_middle['X_policy_premium'] - df_middle['oof_prediction']
df_middle['oof_absolute_error'] = np.abs(df_middle['oof_residual'])

print("\nSample of dataframe with OOF predictions:")
print(df_middle[['id', 'is_churn', 'fold_number', 'oof_prediction', 'oof_residual']].head(10))

print("\nOOF Performance by Fold:")
fold_summary = df_middle.groupby('fold_number').agg({
    'is_churn':            'count',
    'oof_absolute_error':  'mean'
}).rename(columns={'is_churn': 'count', 'oof_absolute_error': 'mae'})
print(fold_summary)


Fold 1/5


[I 2026-04-07 20:17:09,956] A new study created in memory with name: Xgboost
[I 2026-04-07 20:17:15,657] Trial 0 finished with value: 31.45591211562806 and parameters: {'learning_rate': 0.06192300299095742, 'max_depth': 2, 'subsample': 0.3575471836396081, 'colsample_bytree': 0.8858131915291948, 'min_child_weight': 19}. Best is trial 0 with value: 31.45591211562806.
[I 2026-04-07 20:17:18,512] Trial 1 finished with value: 39.04611370320586 and parameters: {'learning_rate': 0.004209220818771749, 'max_depth': 1, 'subsample': 0.07038683062668837, 'colsample_bytree': 0.8164818836543422, 'min_child_weight': 19}. Best is trial 0 with value: 31.45591211562806.
[I 2026-04-07 20:17:22,983] Trial 2 finished with value: 40.284401545971434 and parameters: {'learning_rate': 0.002522159700290059, 'max_depth': 4, 'subsample': 0.40454116178946314, 'colsample_bytree': 0.2027090819939552, 'min_child_weight': 4}. Best is trial 0 with value: 31.45591211562806.
[I 2026-04-07 20:17:27,188] Trial 3 finished w

Best params for fold 1: {'learning_rate': 0.038751445826578565, 'max_depth': 8, 'subsample': 0.9976837478996017, 'colsample_bytree': 0.7319195288717291, 'min_child_weight': 15}


[I 2026-04-07 20:20:11,838] A new study created in memory with name: Xgboost


Fold 1 MAE (log scale):      31.2250
Fold 1 MAE (original scale): 31.2250

Fold 2/5


...

Best params for fold 4: {'learning_rate': 0.06495579081935007, 'max_depth': 7, 'subsample': 0.3443320305211701, 'colsample_bytree': 0.7056473304907893, 'min_child_weight': 11}


[I 2026-04-07 20:28:08,567] A new study created in memory with name: Xgboost


Fold 4 MAE (log scale):      30.6681
Fold 4 MAE (original scale): 30.6681

Fold 5/5


[I 2026-04-07 20:28:11,754] Trial 0 finished with value: 35.059181158977154 and parameters: {'learning_rate': 0.010648547753918039, 'max_depth': 2, 'subsample': 0.16010201852602707, 'colsample_bytree': 0.568168702096513, 'min_child_weight': 6}. Best is trial 0 with value: 35.059181158977154.
[I 2026-04-07 20:28:17,166] Trial 1 finished with value: 31.57692655905534 and parameters: {'learning_rate': 0.018518052631618584, 'max_depth': 7, 'subsample': 0.7133778748611332, 'colsample_bytree': 0.5583320988911838, 'min_child_weight': 11}. Best is trial 1 with value: 31.57692655905534.
[I 2026-04-07 20:28:25,035] Trial 2 finished with value: 32.491519062650724 and parameters: {'learning_rate': 0.030559546038999193, 'max_depth': 2, 'subsample': 0.7763443071602072, 'colsample_bytree': 0.9644217654118817, 'min_child_weight': 1}. Best is trial 1 with value: 31.57692655905534.
[I 2026-04-07 20:28:30,660] Trial 3 finished with value: 36.00467676554572 and parameters: {'learning_rate': 0.003975007678

Best params for fold 5: {'learning_rate': 0.07855875143099925, 'max_depth': 6, 'subsample': 0.8966692662805076, 'colsample_bytree': 0.6399679371719589, 'min_child_weight': 6}
Fold 5 MAE (log scale):      30.6989
Fold 5 MAE (original scale): 30.6989

Cross-Validation Summary
Mean MAE (log scale):      30.8165 (+/- 0.2356)
Mean MAE (original scale): 30.8165 (+/- 0.2356)

Overall OOF MAE: 30.8165

Sample of dataframe with OOF predictions:
       id  is_churn  fold_number  oof_prediction  oof_residual
7       5     False            5      208.090927    -32.318427
16     85     False            4      157.774734     23.093866
24     93     False            1      170.593414     -4.953114
32    119     False            5      173.659164     -2.324464
118   395     False            5      213.783005    -43.684905
139   470     False            3      211.152405    -54.038705
146   478     False            2      179.256470    -27.687570
154   527     False            4      224.995499     18.

## Section 8 — XGBoost Churn Classification (5-Fold OOF)

**Target variable:** `is_churn` (binary, 0 = retained, 1 = churned)  
**Features:** `model_features_class` — 11 features including `X_policy_premium` and `U`

`cross_validate_churn_with_oof_predictions` follows the same nested structure as the regression CV:
1. Outer 5-fold for OOF predictions  
2. Inner 80/20 split for Optuna tuning (maximise ROC-AUC)

**Outputs stored in `df_middle_class`:**
- `fold_number` — fold assignment  
- `oof_prediction` — OOF churn probability (output of `predict_proba[:, 1]`)
- `prob_acceptance = 1 − oof_prediction` — probability the client accepts (does not churn)
- `Z = 1 − is_churn` — binary acceptance indicator

The `prob_acceptance` column is the key output consumed by the pricing optimiser.

In [ ]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

def cross_validate_churn_with_oof_predictions(X, y, n_splits=5, n_trials=30):
    """
    Perform cross-validation and generate out-of-fold predictions.
    FeatureProcessor is fit on each training fold and applied to both train and val
    to prevent data leakage.
    """
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)

    fold_numbers    = np.zeros(len(X))
    oof_predictions = np.zeros(len(X))

    fold_results         = []
    best_params_per_fold = []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'='*50}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'='*50}")

        fold_numbers[val_idx] = fold_idx + 1

        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        # --- fit preprocessor on train fold only (no leakage) ---
        fp = FeatureProcessor()
        X_train_proc = fp.fit_transform(X_train_fold)
        X_val_proc   = fp.transform(X_val_fold)

        from sklearn.model_selection import train_test_split
        X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
            X_train_proc, y_train_fold, test_size=0.2, random_state=123
        )

        model_class = ModelClassifier()
        best_params = model_class.tuning(
            X_train_tune, y_train_tune,
            X_val_tune,   y_val_tune,
            n_trials=n_trials
        )

        print(f"Best params for fold {fold_idx + 1}: {best_params}")
        best_params_per_fold.append(best_params)

        final_model = model_class.train(
            X_train_proc, y_train_fold,
            X_val_proc,   y_val_fold,
            best_params
        )

        y_pred_proba = final_model.predict_proba(X_val_proc)[:, 1]
        roc_auc      = metrics.roc_auc_score(y_true=y_val_fold, y_score=y_pred_proba)

        oof_predictions[val_idx] = y_pred_proba

        fold_results.append({
            'fold':    fold_idx + 1,
            'roc_auc': roc_auc,
            'n_train': len(train_idx),
            'n_val':   len(val_idx)
        })

        print(f"Fold {fold_idx + 1} AUC: {roc_auc:.4f}")

    results_df = pd.DataFrame(fold_results)

    print(f"\n{'='*50}")
    print("Cross-Validation Summary")
    print(f"{'='*50}")
    print(f"Mean AUC: {results_df['roc_auc'].mean():.4f} (+/- {results_df['roc_auc'].std():.4f})")

    return fold_numbers, oof_predictions, results_df, best_params_per_fold


X = df_middle_class.loc[:, model_features_class]
y = df_middle_class['is_churn'].astype(int)

fold_numbers, oof_predictions, results_df, best_params_list = cross_validate_churn_with_oof_predictions(
    X, y, n_splits=5, n_trials=30
)

df_middle_class['fold_number']    = fold_numbers.astype(int)
df_middle_class['oof_prediction'] = oof_predictions


Fold 1/5


[I 2026-04-07 20:31:03,193] A new study created in memory with name: Xgboost
[I 2026-04-07 20:31:04,778] Trial 0 finished with value: 0.6542729977668649 and parameters: {'learning_rate': 0.05195535262891233, 'max_depth': 2, 'subsample': 0.8519703364881129, 'colsample_bytree': 0.29814907290915027, 'min_child_weight': 12}. Best is trial 0 with value: 0.6542729977668649.
[I 2026-04-07 20:31:06,452] Trial 1 finished with value: 0.6443876869707705 and parameters: {'learning_rate': 0.005275566158735326, 'max_depth': 8, 'subsample': 0.05480191038633371, 'colsample_bytree': 0.1882059183412142, 'min_child_weight': 11}. Best is trial 0 with value: 0.6542729977668649.
[I 2026-04-07 20:31:08,326] Trial 2 finished with value: 0.6337223191631345 and parameters: {'learning_rate': 0.0030966705491492115, 'max_depth': 10, 'subsample': 0.5324095732245242, 'colsample_bytree': 0.07194226196334333, 'min_child_weight': 5}. Best is trial 0 with value: 0.6542729977668649.
[I 2026-04-07 20:31:09,516] Trial 3 fi

Best params for fold 1: {'learning_rate': 0.02457868191658342, 'max_depth': 6, 'subsample': 0.1995982026239695, 'colsample_bytree': 0.7517232076681153, 'min_child_weight': 18}


[I 2026-04-07 20:31:54,325] A new study created in memory with name: Xgboost


Fold 1 AUC: 0.6670

Fold 2/5


...

Best params for fold 4: {'learning_rate': 0.0933749763002824, 'max_depth': 3, 'subsample': 0.8854701079544368, 'colsample_bytree': 0.6098335329937612, 'min_child_weight': 1}


[I 2026-04-07 20:34:10,233] A new study created in memory with name: Xgboost


Fold 4 AUC: 0.6611

Fold 5/5


[I 2026-04-07 20:34:11,782] Trial 0 finished with value: 0.6558014053116834 and parameters: {'learning_rate': 0.002078741016752897, 'max_depth': 6, 'subsample': 0.2634200058287854, 'colsample_bytree': 0.2844756709561361, 'min_child_weight': 20}. Best is trial 0 with value: 0.6558014053116834.
[I 2026-04-07 20:34:13,606] Trial 1 finished with value: 0.6404457716195506 and parameters: {'learning_rate': 0.0037275590320679846, 'max_depth': 8, 'subsample': 0.3200342058698116, 'colsample_bytree': 0.06805206506901729, 'min_child_weight': 13}. Best is trial 0 with value: 0.6558014053116834.
[I 2026-04-07 20:34:14,755] Trial 2 finished with value: 0.641653253982688 and parameters: {'learning_rate': 0.0023717557835721535, 'max_depth': 2, 'subsample': 0.06713136189665113, 'colsample_bytree': 0.4446655840994831, 'min_child_weight': 7}. Best is trial 0 with value: 0.6558014053116834.
[I 2026-04-07 20:34:15,781] Trial 3 finished with value: 0.604892756865979 and parameters: {'learning_rate': 0.00112

Best params for fold 5: {'learning_rate': 0.04237893606534701, 'max_depth': 5, 'subsample': 0.745076604162433, 'colsample_bytree': 0.498014838276501, 'min_child_weight': 8}
Fold 5 AUC: 0.6706

Cross-Validation Summary
Mean AUC: 0.6661 (+/- 0.0040)


In [283]:
df_middle_class.U.describe()

count    194373.000000
mean          0.083095
std           0.029994
min          -0.001829
25%           0.061570
50%           0.094024
75%           0.103913
max           0.418037
Name: U, dtype: float64

## Section 9 — XGBoost Output: Acceptance Probability & Expected Financial Loss

Post-CV, the following export steps are performed:

### `df_acceptance_xgb_black_box.csv`
Selected columns from `df_middle_class`:  
`id, X_age, X_bonus_malus_rating, X_distr_channel, X_vehicle_type, X_ttm_claims, X_policy_count, X_risk_code, X_vehicle_age, X_policy_tenure, X_policy_premium, U, prob_acceptance, Z`

- `prob_acceptance` = acceptance probability predicted by the XGBoost classifier  
- `U` = premium uplift factor (`1 + relative_change`)  
- `Z` = binary acceptance outcome (ground truth)

### `df_exp_financial_loss_xgb_black_box.csv`
Selected columns from `df_middle`:  
`id, ..., Y (actual premium), U, Y_hat (predicted premium)`

- `Y_hat` = XGBoost regressor's OOF premium prediction  
- Used to compute expected financial loss = `Y_hat × (1 − prob_acceptance)`

In [284]:
df_middle_class['Z'] = 1- df_middle_class['is_churn'].astype(int)

In [285]:
model_features_class

['X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure',
 'X_policy_premium',
 'U']

In [286]:
df_middle_class['prob_acceptance'] = 1 - df_middle_class['oof_prediction']

In [287]:
df_middle_class.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'U', 'fold_number', 'oof_prediction', 'Z',
       'prob_acceptance'],
      dtype='str')

In [288]:
df_middle.columns = ['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                     'Y','X_prev_renewal_perc','X_upcoming_premium','X_cur_renewal_perc','X_year','1-Z','U','fold_number','Y_hat','Y_residual','Y_absolute_error']

In [289]:
df_middle_class.columns = ['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code',"X_vehicle_age",
                           'X_policy_tenure','X_policy_premium','X_prev_renewal_perc','X_upcoming_premium','X_cur_renewal_perc','X_year','1-Z', 'U','fold_number','churn_prediction','Z','prob_acceptance']

In [290]:
df_middle_class['U+1'] = 1 + df_middle_class['U']

In [291]:
df_middle_class.loc[:, ['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age',
                           'X_policy_tenure','X_policy_premium', 'U','prob_acceptance','Z']]

,id,X_age,X_bonus_malus_rating,X_distr_channel,X_vehicle_type,X_ttm_claims,X_policy_count,X_risk_code,X_vehicle_age,X_policy_tenure,X_policy_premium,U,prob_acceptance,Z
7,5,93.0,45.0,5,7,0,1,0,24.0,24.0,175.7725,0.123099,0.848508,1
16,85,78.0,45.0,5,0,0,2,0,5.0,24.0,180.8686,0.107047,0.946727,1
24,93,78.0,45.0,5,1,0,1,0,21.0,24.0,165.6403,0.096895,0.920173,1
32,119,49.0,45.0,5,1,0,1,0,19.0,24.0,171.3347,0.097209,0.931579,1
118,395,49.0,45.0,2,2,0,4,0,16.0,24.0,170.0981,0.102011,0.941084,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3370174,5732129,68.0,45.0,4,2,0,1,2,9.0,9.0,345.4078,0.062686,0.930541,1
3370323,5732370,81.0,45.0,4,4,0,5,0,20.0,10.0,234.6113,0.087288,0.949478,1
3370329,5732373,58.0,45.0,4,7,0,3,0,30.0,8.0,174.1869,0.087051,0.918007,0
3370407,5732452,89.0,45.0,4,1,0,2,0,10.0,23.0,238.8796,0.087493,0.927157,0


In [292]:
df_middle_class.loc[:, ['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age',
                           'X_policy_tenure','X_policy_premium', 'U', 'U+1','prob_acceptance','Z']].to_csv('df_acceptance_xgb_black_box_feat_processor.csv',sep=';')

In [293]:
df_middle.loc[:,['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                     'Y','U','Y_hat']]

,id,X_age,X_bonus_malus_rating,X_distr_channel,X_vehicle_type,X_ttm_claims,X_policy_count,X_risk_code,X_vehicle_age,X_policy_tenure,Y,U,Y_hat
7,5,93.0,45.0,5,7,0,1,0,24.0,24.0,175.7725,0.123099,208.090927
16,85,78.0,45.0,5,0,0,2,0,5.0,24.0,180.8686,0.107047,157.774734
24,93,78.0,45.0,5,1,0,1,0,21.0,24.0,165.6403,0.096895,170.593414
32,119,49.0,45.0,5,1,0,1,0,19.0,24.0,171.3347,0.097209,173.659164
118,395,49.0,45.0,2,2,0,4,0,16.0,24.0,170.0981,0.102011,213.783005
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3370174,5732129,68.0,45.0,4,2,0,1,2,9.0,9.0,345.4078,0.062686,261.505676
3370323,5732370,81.0,45.0,4,4,0,5,0,20.0,10.0,234.6113,0.087288,231.815430
3370329,5732373,58.0,45.0,4,7,0,3,0,30.0,8.0,174.1869,0.087051,191.933762
3370407,5732452,89.0,45.0,4,1,0,2,0,10.0,23.0,238.8796,0.087493,184.055298


In [294]:
df_middle['U+1'] = 1 + df_middle['U']

In [295]:
df_middle.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'Y', 'X_prev_renewal_perc',
       'X_upcoming_premium', 'X_cur_renewal_perc', 'X_year', '1-Z', 'U',
       'fold_number', 'Y_hat', 'Y_residual', 'Y_absolute_error', 'U+1'],
      dtype='str')

In [296]:
df_middle.loc[:,['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                     'Y','U','U+1','Y_hat']].to_csv('df_exp_financial_loss_xgb_black_box_feat_processor.csv',sep=';')

In [297]:
from pygam import LinearGAM, s
from pygam.datasets import toy_interaction

X, y = toy_interaction(return_X_y=True)

gam = LinearGAM(s(0, by=1)).fit(X, y)
gam.summary()

LinearGAM                                                                                                 
=============================================== ==========================================================
Distribution:                        NormalDist Effective DoF:                                     20.8506
Link Function:                     IdentityLink Log Likelihood:                                 44202.0909
Number of Samples:                        50000 AIC:                                           -88360.4805
                                                AICc:                                          -88360.4605
                                                GCV:                                                  0.01
                                                Scale:                                                 0.1
                                                Pseudo R-Squared:                                   0.9976
Feature Function                  Lam

C:\Users\malosett\AppData\Local\Temp\ipykernel_13900\427681795.py:7: UserWarning: KNOWN BUG: p-values computed in this summary are likely much smaller than they should be. 
 
Please do not make inferences based on these values! 

Collaborate on a solution, and stay up to date at: 
github.com/dswah/pyGAM/issues/163 

  gam.summary()


In [298]:
!pip install pygam

Defaulting to user installation because normal site-packages is not writeable


## Section 10 — GLM & GAM Dataset Preparation

Three additional datasets are created for the interpretable models, all using the same `filter_middle_advanced` thresholds:

| Dataset | Used by |
|---|---|
| `df_middle_glm` | GLM (Logistic Regression) churn classifier |
| `df_middle_gam` | GAM (LogisticGAM) churn classifier |
| `df_middle_reg` | (supporting dataset, currently superseded by `df_middle_reg_linear`) |

For GAM and GLM, categorical columns are **label-encoded** so that the preprocessors within each pipeline start from the same integer representation.  
`U` is computed and added to each dataset before training.

In [299]:
df_middle_glm = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [300]:
df_middle_glm['U'] = (df_middle_glm['X_upcoming_premium'] - df_middle_glm['X_policy_premium']) / df_middle_glm['X_policy_premium']

In [301]:
df_middle_gam = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [302]:
df_middle_gam.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='str')

In [303]:
df_middle_gam['U'] = (df_middle_gam['X_upcoming_premium'] - df_middle_gam['X_policy_premium']) / df_middle_gam['X_policy_premium']

In [304]:
df_middle_reg = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [305]:
df_middle_glm

,id,X_age,X_bonus_malus_rating,X_distr_channel,X_vehicle_type,X_ttm_claims,X_policy_count,X_risk_code,X_vehicle_age,X_policy_tenure,X_policy_premium,X_prev_renewal_perc,X_upcoming_premium,X_cur_renewal_perc,year,is_churn,U
7,5,93.0,45.0,19,T4005040,0,1,0,24.0,24.0,175.7725,0.003637,197.41,0.098601,7,False,0.123099
16,85,78.0,45.0,19,T4003010,0,2,0,5.0,24.0,180.8686,0.003824,200.23,0.082852,7,False,0.107047
24,93,78.0,45.0,19,T4003020,0,1,0,21.0,24.0,165.6403,0.003731,181.69,0.072973,7,False,0.096895
32,119,49.0,45.0,19,T4003020,0,1,0,19.0,24.0,171.3347,-0.005503,187.99,0.073266,7,False,0.097209
118,395,49.0,45.0,11,T4003030,0,4,0,16.0,24.0,170.0981,-0.005223,187.45,0.077961,7,False,0.102011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3370174,5732129,68.0,45.0,16,T4003030,0,1,11,9.0,9.0,345.4078,-0.023380,367.06,0.042169,7,False,0.062686
3370323,5732370,81.0,45.0,16,T4003050,0,5,0,20.0,10.0,234.6113,-0.040569,255.09,0.066399,7,False,0.087288
3370329,5732373,58.0,45.0,16,T4005040,0,3,0,30.0,8.0,174.1869,-0.074677,189.35,0.061103,7,True,0.087051
3370407,5732452,89.0,45.0,16,T4003020,0,2,0,10.0,23.0,238.8796,-0.060040,259.78,0.064402,7,True,0.087493


In [306]:
df_middle_reg['U'] = (df_middle_reg['X_upcoming_premium'] - df_middle_reg['X_policy_premium']) / df_middle_reg['X_policy_premium']

In [307]:
from sklearn.preprocessing import LabelEncoder
for cat in cat_cols:
    le = LabelEncoder()
    le.fit(df_middle_glm[cat])
    df_middle_glm[cat] = le.transform(df_middle_glm[cat])
    df_middle_gam[cat] = le.transform(df_middle_gam[cat])
    df_middle_reg[cat] = le.transform(df_middle_reg[cat])


## Section 11 — GLM Churn Cross-Validation (Logistic Regression)

### `ModelGLM`
A **sklearn `Pipeline`** wrapping:
1. `ColumnTransformer` — median imputation + standard scaling for numerics; mode imputation + one-hot encoding for categoricals
2. `LogisticRegression` (solver: `lbfgs`, max_iter: 1000)

**Tuning:** Optuna over the regularisation parameter `C` (log-uniform in `[0.001, 100]`), maximising ROC-AUC.

### `cross_validate_glm_with_oof`
- 5-fold outer CV with an inner 80/20 tuning split per fold  
- Outputs OOF churn probabilities (`oof_prediction_glm`) and the fold assignment

**Why GLM?**  
Provides an interpretable, linear-in-log-odds alternative to XGBoost.  
Suitable when a regulator requires explainability of acceptance probability estimates.

In [308]:
from sklearn import metrics
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
import optuna
from optuna.trial import Trial


class ModelGLM():
    """
    GLM-like classifier implemented with sklearn LogisticRegression.
    This avoids statsmodels exog shape mismatches across folds.
    """

    def __init__(self):
        self.categorical_cols = []
        self.numeric_cols = []

    def _build_model(self, C: float) -> Pipeline:
        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])

        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, self.numeric_cols),
                ("cat", categorical_transformer, self.categorical_cols),
            ],
            remainder="drop",
        )

        clf = LogisticRegression(
            C=C,
            max_iter=1000,
            solver="lbfgs",
            random_state=123,
        )

        model = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf),
        ])
        return model

    def objective(self, trial: Trial, X_train: pd.DataFrame, y_train: pd.Series,
                  X_val: pd.DataFrame, y_val: pd.Series) -> float:
        """Optuna objective for logistic regression."""
        C = trial.suggest_float("C", 1e-3, 100.0, log=True)

        # Infer schema from training fold only.
        self.categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
        self.numeric_cols = [c for c in X_train.columns if c not in self.categorical_cols]

        try:
            model = self._build_model(C=C)
            model.fit(X_train, y_train)
            y_score = model.predict_proba(X_val)[:, 1]
            y_score = np.clip(y_score, 1e-10, 1 - 1e-10)
            roc_auc = metrics.roc_auc_score(y_true=y_val, y_score=y_score)
            return roc_auc
        except Exception as e:
            print(f"Error in LogisticRegression fitting: {e}")
            return 0.5

    def tuning(self, X_train: pd.DataFrame, y_train: pd.Series,
               X_val: pd.DataFrame, y_val: pd.Series, n_trials: int) -> dict:
        """Hyperparameter tuning using Optuna."""
        study = optuna.create_study(study_name="LogisticRegression", direction="maximize")
        study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=n_trials,
            show_progress_bar=True,
        )
        return study.best_params

    def train(self, X_train: pd.DataFrame, y_train: pd.Series,
              X_val: pd.DataFrame, y_val: pd.Series, best_params: dict):
        """Train final logistic regression with best parameters."""
        C = best_params.get("C", 1.0)
        self.categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
        self.numeric_cols = [c for c in X_train.columns if c not in self.categorical_cols]

        model = self._build_model(C=C)
        model.fit(X_train, y_train)
        return model


def cross_validate_glm_with_oof(X, y, n_splits=5, n_trials=20):
    """
    Cross-validation with OOF predictions for Logistic Regression GLM.
    FeatureProcessor is fit on each training fold and applied to both train and val
    to prevent data leakage. U is kept untouched (not processed).
    """
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)

    fold_numbers = np.zeros(len(X))
    oof_predictions_proba = np.zeros(len(X))
    oof_predictions_class = np.zeros(len(X))

    fold_results = []
    best_params_per_fold = []

    # Identify columns to keep untouched (U)
    u_cols = [col for col in ["U"] if col in X.columns]
    x_feature_cols = [col for col in X.columns if col not in u_cols]

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'='*70}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'='*70}")

        fold_numbers[val_idx] = fold_idx + 1

        X_train_fold = X.iloc[train_idx].reset_index(drop=True)
        X_val_fold = X.iloc[val_idx].reset_index(drop=True)
        y_train_fold = y.iloc[train_idx].reset_index(drop=True)
        y_val_fold = y.iloc[val_idx].reset_index(drop=True)

        # --- Separate U from other features ---
        if u_cols:
            U_train = X_train_fold[u_cols].copy()
            U_val = X_val_fold[u_cols].copy()
            X_train_raw = X_train_fold[x_feature_cols].copy()
            X_val_raw = X_val_fold[x_feature_cols].copy()
        else:
            X_train_raw = X_train_fold.copy()
            X_val_raw = X_val_fold.copy()

        # --- Fit preprocessor on train fold only (no leakage) ---
        fp = FeatureProcessor()
        X_train_proc = fp.fit_transform(X_train_raw)
        X_val_proc = fp.transform(X_val_raw)

        # --- Concatenate U back with processed features ---
        if u_cols:
            X_train_proc = pd.concat([U_train.reset_index(drop=True), X_train_proc.reset_index(drop=True)], axis=1)
            X_val_proc = pd.concat([U_val.reset_index(drop=True), X_val_proc.reset_index(drop=True)], axis=1)

        # Split for tuning
        X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
            X_train_proc, y_train_fold, test_size=0.2, random_state=123, stratify=y_train_fold
        )

        # Tune
        model_glm = ModelGLM()
        best_params = model_glm.tuning(
            X_train_tune, y_train_tune,
            X_val_tune, y_val_tune,
            n_trials=n_trials,
        )

        print(f"Best params: {best_params}")
        best_params_per_fold.append(best_params)

        # Train
        final_model = model_glm.train(
            X_train_proc, y_train_fold,
            X_val_proc, y_val_fold,
            best_params,
        )

        # Predict
        y_pred_proba = final_model.predict_proba(X_val_proc)[:, 1]
        y_pred_proba = np.clip(y_pred_proba, 0, 1)
        y_pred_class = (y_pred_proba > 0.5).astype(int)

        oof_predictions_proba[val_idx] = y_pred_proba
        oof_predictions_class[val_idx] = y_pred_class

        # Metrics
        roc_auc = metrics.roc_auc_score(y_val_fold, y_pred_proba)
        accuracy = metrics.accuracy_score(y_val_fold, y_pred_class)

        fold_results.append({
            "fold": fold_idx + 1,
            "roc_auc": roc_auc,
            "accuracy": accuracy,
            "n_train": len(train_idx),
            "n_val": len(val_idx),
        })

        print(f"Fold {fold_idx + 1} ROC-AUC: {roc_auc:.4f}")
        print(f"Fold {fold_idx + 1} Accuracy: {accuracy:.4f}")

    results_df = pd.DataFrame(fold_results)

    print(f"\n{'='*70}")
    print("Cross-Validation Summary")
    print(f"{'='*70}")
    print(f"Mean ROC-AUC: {results_df['roc_auc'].mean():.4f} (+/- {results_df['roc_auc'].std():.4f})")
    print(f"Mean Accuracy: {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")

    overall_roc_auc = metrics.roc_auc_score(y, oof_predictions_proba)
    print(f"\nOverall OOF ROC-AUC: {overall_roc_auc:.4f}")

    return fold_numbers, oof_predictions_proba, results_df, best_params_per_fold


# Usage
X = df_middle_glm.loc[:, model_features_class]
y = df_middle_glm['is_churn']

fold_numbers_glm, oof_pred_glm, results_glm, params_glm = cross_validate_glm_with_oof(
    X, y, n_splits=5, n_trials=20
)

df_middle_glm['fold_number_glm'] = fold_numbers_glm.astype(int)
df_middle_glm['oof_prediction_glm'] = oof_pred_glm


Fold 1/5


[I 2026-04-07 20:35:11,630] A new study created in memory with name: LogisticRegression


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-07 20:35:12,100] Trial 0 finished with value: 0.651126863475761 and parameters: {'C': 2.0215005819761416}. Best is trial 0 with value: 0.651126863475761.
[I 2026-04-07 20:35:12,463] Trial 1 finished with value: 0.6511312227057081 and parameters: {'C': 0.060295279198897773}. Best is trial 1 with value: 0.6511312227057081.
[I 2026-04-07 20:35:12,836] Trial 2 finished with value: 0.6511304218239272 and parameters: {'C': 0.0796522825530112}. Best is trial 1 with value: 0.6511312227057081.
[I 2026-04-07 20:35:13,219] Trial 3 finished with value: 0.6511912989770255 and parameters: {'C': 0.003636189860693768}. Best is trial 3 with value: 0.6511912989770255.
[I 2026-04-07 20:35:13,668] Trial 4 finished with value: 0.651130411686183 and parameters: {'C': 0.08001387052396138}. Best is trial 3 with value: 0.6511912989770255.
[I 2026-04-07 20:35:14,043] Trial 5 finished with value: 0.6511268736135051 and parameters: {'C': 3.293768263682318}. Best is trial 3 with value: 0.651191298977025

[I 2026-04-07 20:35:19,539] A new study created in memory with name: LogisticRegression


...

[I 2026-04-07 20:35:33,084] Trial 0 finished with value: 0.6437396783462526 and parameters: {'C': 71.16287606669546}. Best is trial 0 with value: 0.6437396783462526.
[I 2026-04-07 20:35:33,354] Trial 1 finished with value: 0.6438024425263343 and parameters: {'C': 0.0036813378496365684}. Best is trial 1 with value: 0.6438024425263343.
[I 2026-04-07 20:35:33,617] Trial 2 finished with value: 0.6437397499015067 and parameters: {'C': 3.3075357188290586}. Best is trial 1 with value: 0.6438024425263343.
[I 2026-04-07 20:35:33,874] Trial 3 finished with value: 0.6437419578922067 and parameters: {'C': 0.10232580719289011}. Best is trial 1 with value: 0.6438024425263343.
[I 2026-04-07 20:35:34,137] Trial 4 finished with value: 0.6437654177933936 and parameters: {'C': 0.009685360750226385}. Best is trial 1 with value: 0.6438024425263343.
[I 2026-04-07 20:35:34,406] Trial 5 finished with value: 0.6437506365223189 and parameters: {'C': 0.02585223288638243}. Best is trial 1 with value: 0.6438024425

[I 2026-04-07 20:35:39,014] A new study created in memory with name: LogisticRegression


Fold 4 ROC-AUC: 0.6394
Fold 4 Accuracy: 0.8809

Fold 5/5


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-07 20:35:39,286] Trial 0 finished with value: 0.6459718692560501 and parameters: {'C': 3.7216439749167174}. Best is trial 0 with value: 0.6459718692560501.
[I 2026-04-07 20:35:39,551] Trial 1 finished with value: 0.645971362122769 and parameters: {'C': 0.5541071355718894}. Best is trial 0 with value: 0.6459718692560501.
[I 2026-04-07 20:35:39,815] Trial 2 finished with value: 0.645971788114725 and parameters: {'C': 1.7645874222233446}. Best is trial 0 with value: 0.6459718692560501.
[I 2026-04-07 20:35:40,096] Trial 3 finished with value: 0.6459718591133843 and parameters: {'C': 0.050907844907920115}. Best is trial 0 with value: 0.6459718692560501.
[I 2026-04-07 20:35:40,362] Trial 4 finished with value: 0.6459718286853875 and parameters: {'C': 11.304211013820058}. Best is trial 0 with value: 0.6459718692560501.
[I 2026-04-07 20:35:40,624] Trial 5 finished with value: 0.6459732486585741 and parameters: {'C': 0.0010025121848850898}. Best is trial 5 with value: 0.6459732486585

In [309]:
df_middle_glm.head()


,id,X_age,X_bonus_malus_rating,X_distr_channel,X_vehicle_type,X_ttm_claims,X_policy_count,X_risk_code,X_vehicle_age,X_policy_tenure,X_policy_premium,X_prev_renewal_perc,X_upcoming_premium,X_cur_renewal_perc,year,is_churn,U,fold_number_glm,oof_prediction_glm
7,5,93.0,45.0,5,7,0,1,0,24.0,24.0,175.7725,0.003637,197.41,0.098601,7,False,0.123099,5,0.134916
16,85,78.0,45.0,5,0,0,2,0,5.0,24.0,180.8686,0.003824,200.23,0.082852,7,False,0.107047,4,0.049394
24,93,78.0,45.0,5,1,0,1,0,21.0,24.0,165.6403,0.003731,181.69,0.072973,7,False,0.096895,1,0.089371
32,119,49.0,45.0,5,1,0,1,0,19.0,24.0,171.3347,-0.005503,187.99,0.073266,7,False,0.097209,5,0.100781
118,395,49.0,45.0,2,2,0,4,0,16.0,24.0,170.0981,-0.005223,187.45,0.077961,7,False,0.102011,5,0.067738


## Section 12 — GLM Output: Acceptance Probability

Post-CV steps for `df_middle_glm`:

1. Compute `Z = 1 − is_churn` (binary acceptance)  
2. Compute `prob_acceptance = 1 − oof_prediction_glm`  
3. Adjust `U = 1 + U` (uplift factor format)  
4. Rename columns to canonical business names  

**Exported to:** `df_acceptance_linear_model_black_box.csv`

This file mirrors the XGBoost acceptance output but with GLM-derived acceptance probabilities, allowing the optimisation module to compare model choices.

In [310]:
df_middle_glm['Z'] = 1 - df_middle_glm['is_churn'].astype(int)
df_middle_glm['prob_acceptance'] = 1 - df_middle_glm['oof_prediction_glm']

In [311]:
df_middle_glm['U+1'] = 1 + df_middle_glm['U']

In [312]:
df_middle_glm.columns = ['id','X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                         'X_policy_premium','X_prev_renewal_perc','X_upcoming_premium','X_cur_renewal_perc','X_year','1-Z','U','fold_number_glm','churn_prediction','Z','prob_acceptance','U+1']

In [313]:
df_middle_glm.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'X_year', '1-Z', 'U', 'fold_number_glm', 'churn_prediction', 'Z',
       'prob_acceptance', 'U+1'],
      dtype='str')

In [314]:
df_middle_glm.to_csv('df_acceptance_linear_model_black_box_feat_processor.csv',sep=';')

In [315]:
df_middle_gam.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'U'],
      dtype='str')

## Section 13 — GAM Churn Cross-Validation (LogisticGAM)

### `ModelGAM` (memory-safe version)
A **pyGAM `LogisticGAM`** that uses:
- **Spline terms** (`s()`) for continuous features — captures non-linear relationships
- **Factor terms** (`f()`) for categorical features — discrete category effects

Key design decisions:
- `_prepare_data` — category mappings are fitted on training data only (prevents leakage)
- Progressively tries `n_splines = [8, 6, 4]` if fitting fails due to memory pressure
- `max_train_samples = 35,000` — stratified subsampling is applied to avoid OOM on large folds

### `cross_validate_gam_with_oof`
- 5-fold outer CV (no inner Optuna tuning; `n_splines` and `lam` are fixed hyperparameters)
- `_stratified_sample` ensures class balance is preserved after subsampling

**Why GAM?**  
GAMs are semi-parametric: they capture non-linear covariate effects while remaining interpretable via partial dependence plots, making them a strong regulatory-friendly alternative.

In [316]:
from sklearn import metrics
import pandas as pd
import numpy as np
from pygam import LogisticGAM, s, f
from sklearn.model_selection import KFold


class ModelGAM:
    """Memory-safe GAM classifier for churn prediction."""

    def __init__(self):
        self.numeric_cols = []
        self.categorical_cols = []
        self.feature_order = []
        self.category_maps = {}
        self.category_default_code = {}

    def _prepare_data(self, X: pd.DataFrame, fit: bool = False) -> pd.DataFrame:
        """Prepare data for GAM using stable category mappings and fixed column order."""
        X_prep = X.copy()

        if fit:
            self.numeric_cols = X_prep.select_dtypes(include=[np.number]).columns.tolist()
            self.categorical_cols = X_prep.select_dtypes(include=["object", "category"]).columns.tolist()
            self.feature_order = self.numeric_cols + self.categorical_cols

            self.category_maps = {}
            self.category_default_code = {}
            for col in self.categorical_cols:
                cats = sorted(X_prep[col].astype(str).dropna().unique().tolist())
                mapping = {cat: idx for idx, cat in enumerate(cats)}
                self.category_maps[col] = mapping
                self.category_default_code[col] = len(mapping)

        for col in self.categorical_cols:
            mapping = self.category_maps[col]
            default_code = self.category_default_code[col]
            X_prep[col] = X_prep[col].astype(str).map(lambda x: mapping.get(x, default_code))

        if len(self.numeric_cols) > 0:
            X_prep[self.numeric_cols] = X_prep[self.numeric_cols].apply(
                lambda s_col: s_col.fillna(s_col.median())
            )

        X_prep = X_prep[self.feature_order]
        return X_prep

    def _build_formula(self, n_features: int, n_numeric: int, n_splines: int = 8, lam: float = 0.8):
        if n_features == 0:
            raise ValueError("No features found for GAM model.")

        terms = None

        for i in range(n_numeric):
            term = s(i, n_splines=n_splines, lam=lam)
            terms = term if terms is None else terms + term

        for i in range(n_numeric, n_features):
            term = f(i, lam=lam)
            terms = term if terms is None else terms + term

        return terms

    def train(
        self,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_val: pd.DataFrame,
        y_val: pd.Series,
        n_splines: int = 8,
        lam: float = 0.8,
    ):
        X_train_prep = self._prepare_data(X_train, fit=True)
        _ = self._prepare_data(X_val, fit=False)

        n_features = X_train_prep.shape[1]
        n_numeric = len(self.numeric_cols)

        # Try progressively simpler models if memory pressure occurs.
        candidate_splines = [n_splines, min(6, n_splines), 4]

        last_error = None
        for spl in candidate_splines:
            try:
                terms = self._build_formula(n_features, n_numeric, n_splines=spl, lam=lam)
                gam = LogisticGAM(terms=terms, max_iter=100)
                gam.fit(X_train_prep.values.astype(np.float32), y_train.values)
                return gam
            except Exception as e:
                last_error = e
                print(f"GAM fit failed with n_splines={spl}: {e}")

        raise RuntimeError(f"All GAM fits failed. Last error: {last_error}")


def _stratified_sample(X: pd.DataFrame, y: pd.Series, max_samples: int, seed: int):
    """Return stratified sample indices for memory-safe GAM fitting."""
    if len(X) <= max_samples:
        return X.reset_index(drop=True), y.reset_index(drop=True)

    rng = np.random.default_rng(seed)
    y_arr = y.values

    pos_idx = np.where(y_arr == 1)[0]
    neg_idx = np.where(y_arr == 0)[0]

    pos_ratio = len(pos_idx) / len(y_arr)
    n_pos = int(max_samples * pos_ratio)
    n_neg = max_samples - n_pos

    n_pos = min(n_pos, len(pos_idx))
    n_neg = min(n_neg, len(neg_idx))

    sampled_pos = rng.choice(pos_idx, size=n_pos, replace=False) if n_pos > 0 else np.array([], dtype=int)
    sampled_neg = rng.choice(neg_idx, size=n_neg, replace=False) if n_neg > 0 else np.array([], dtype=int)

    sample_idx = np.concatenate([sampled_pos, sampled_neg])
    rng.shuffle(sample_idx)

    return X.iloc[sample_idx].reset_index(drop=True), y.iloc[sample_idx].reset_index(drop=True)


def cross_validate_gam_with_oof(
    X,
    y,
    n_splits=5,
    n_splines=8,
    lam=0.8,
    max_train_samples=35000,
):
    """Cross-validation for GAM with OOF predictions and RAM-safe training."""
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)

    fold_numbers = np.zeros(len(X))
    oof_predictions_proba = np.zeros(len(X))
    oof_predictions_class = np.zeros(len(X))

    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'=' * 70}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'=' * 70}")

        fold_numbers[val_idx] = fold_idx + 1

        X_train_fold = X.iloc[train_idx].reset_index(drop=True)
        X_val_fold = X.iloc[val_idx].reset_index(drop=True)
        y_train_fold = y.iloc[train_idx].reset_index(drop=True)
        y_val_fold = y.iloc[val_idx].reset_index(drop=True)

        X_train_fit, y_train_fit = _stratified_sample(
            X_train_fold,
            y_train_fold,
            max_samples=max_train_samples,
            seed=123 + fold_idx,
        )

        print(f"Training rows used: {len(X_train_fit):,} / {len(X_train_fold):,}")

        model_gam = ModelGAM()
        try:
            gam_model = model_gam.train(
                X_train_fit,
                y_train_fit,
                X_val_fold,
                y_val_fold,
                n_splines=n_splines,
                lam=lam,
            )

            X_val_prep = model_gam._prepare_data(X_val_fold, fit=False)
            y_pred_proba = gam_model.predict_proba(X_val_prep.values.astype(np.float32))
            y_pred_class = (y_pred_proba > 0.5).astype(int)

            oof_predictions_proba[val_idx] = y_pred_proba
            oof_predictions_class[val_idx] = y_pred_class

            roc_auc = metrics.roc_auc_score(y_val_fold, y_pred_proba)
            accuracy = metrics.accuracy_score(y_val_fold, y_pred_class)

            fold_results.append(
                {
                    "fold": fold_idx + 1,
                    "roc_auc": roc_auc,
                    "accuracy": accuracy,
                    "n_train_used": len(X_train_fit),
                    "n_train_full": len(train_idx),
                    "n_val": len(val_idx),
                }
            )

            print(f"Fold {fold_idx + 1} ROC-AUC: {roc_auc:.4f}")
            print(f"Fold {fold_idx + 1} Accuracy: {accuracy:.4f}")

        except Exception as e:
            print(f"Error in fold {fold_idx + 1}: {e}")
            oof_predictions_proba[val_idx] = 0.5
            oof_predictions_class[val_idx] = 0

    results_df = pd.DataFrame(fold_results)

    print(f"\n{'=' * 70}")
    print("Cross-Validation Summary")
    print(f"{'=' * 70}")
    if len(results_df) > 0:
        print(
            f"Mean ROC-AUC: {results_df['roc_auc'].mean():.4f} "
            f"(+/- {results_df['roc_auc'].std():.4f})"
        )
        print(
            f"Mean Accuracy: {results_df['accuracy'].mean():.4f} "
            f"(+/- {results_df['accuracy'].std():.4f})"
        )

        overall_roc_auc = metrics.roc_auc_score(y, oof_predictions_proba)
        print(f"\nOverall OOF ROC-AUC: {overall_roc_auc:.4f}")

    return fold_numbers, oof_predictions_proba, results_df


# Usage with your data
X = df_middle_gam.loc[:, model_features_class]
y = df_middle_gam["is_churn"]

print("Running GAM Cross-Validation...")
fold_numbers_gam, oof_pred_gam, results_gam = cross_validate_gam_with_oof(
    X,
    y,
    n_splits=5,
    n_splines=8,
    lam=0.8,
    max_train_samples=35000,
)

df_middle_gam["fold_number_gam"] = fold_numbers_gam.astype(int)
df_middle_gam["oof_prediction_gam"] = oof_pred_gam

Running GAM Cross-Validation...

Fold 1/5
Training rows used: 35,000 / 155,498
Fold 1 ROC-AUC: 0.6622
Fold 1 Accuracy: 0.8854

Fold 2/5
Training rows used: 35,000 / 155,498
Fold 2 ROC-AUC: 0.6599
Fold 2 Accuracy: 0.8858

Fold 3/5
Training rows used: 35,000 / 155,498
Fold 3 ROC-AUC: 0.6520
Fold 3 Accuracy: 0.8870

Fold 4/5
Training rows used: 35,000 / 155,499
Fold 4 ROC-AUC: 0.6575
Fold 4 Accuracy: 0.8810

Fold 5/5
Training rows used: 35,000 / 155,499
Fold 5 ROC-AUC: 0.6640
Fold 5 Accuracy: 0.8851

Cross-Validation Summary
Mean ROC-AUC: 0.6591 (+/- 0.0047)
Mean Accuracy: 0.8849 (+/- 0.0023)

Overall OOF ROC-AUC: 0.6590


In [317]:
import pygam

In [318]:
# Update narwhals to the latest version
!pip install --upgrade narwhals

# Or if that doesn't work, try reinstalling both packages
!pip install --upgrade glum narwhals


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [319]:
df_middle_gam.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'U', 'fold_number_gam', 'oof_prediction_gam'],
      dtype='str')

In [320]:
df_middle_gam.columns = ['id','X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims',
                         'X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure','X_policy_premium',
                         'X_prev_renewal_perc','X_upcoming_premium','X_cur_renewal_perc','X_year','1-Z','U',
                         'fold_number_gam','churn_prediction']

## Section 14 — GAM Output: Acceptance Probability

Post-CV steps for `df_middle_gam`:

1. Compute `Z = 1 − churn_prediction` (binary acceptance, where `churn_prediction = oof_prediction_gam`)  
2. Compute `prob_acceptance = 1 − churn_prediction`  
3. Rename columns to canonical names  

**Exported to:** `df_acceptance_gam_model_black_box.csv`

Same schema as the XGBoost and GLM acceptance files — the optimiser can swap between models with no interface change.

In [321]:
df_middle_gam['Z'] = 1 - df_middle_gam['1-Z'].astype(int)
df_middle_gam['prob_acceptance'] = 1 - df_middle_gam['churn_prediction']

In [322]:
df_middle_gam['U+1'] = 1 + df_middle_gam['U']

In [323]:
df_middle_gam.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'X_year', '1-Z', 'U', 'fold_number_gam', 'churn_prediction', 'Z',
       'prob_acceptance', 'U+1'],
      dtype='str')

In [324]:
df_middle_gam.loc[:, ['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age',
                           'X_policy_tenure','X_policy_premium', 'U','prob_acceptance','Z','U+1']].to_csv('df_acceptance_gam_model_black_box_feat_processor.csv',sep=';')

In [325]:
df_middle_gam['U+1'] = 1 + df_middle_gam['U']

In [326]:
from sklearn import metrics
import pandas as pd
import numpy as np
from pygam import LogisticGAM, s, f, te
from sklearn.model_selection import KFold, train_test_split
import optuna
from optuna.trial import Trial

class ModelGAM():
    """
    GAM Classifier for Churn Prediction
    GAMs are interpretable and can capture non-linear relationships
    """
    
    def objective(self, trial: Trial, X_train: pd.DataFrame, y_train: pd.Series, 
                  X_val: pd.DataFrame, y_val: pd.Series) -> float:
        """
        Optuna objective for hyperparameter tuning
        """
        # Suggest hyperparameters
        n_splines = trial.suggest_int("n_splines", 10, 50)
        lam = trial.suggest_float("lam", 0.01, 100, log=True)
        
        # Build GAM formula
        # Use splines for continuous features and factors for categorical
        numeric_cols = X_train.select_dtypes(include=[np.number]).columns
        categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
        
        # Create formula
        formula_terms = []
        for i, col in enumerate(numeric_cols):
            formula_terms.append(s(i, n_splines=n_splines, lam=lam))
        
        # For categorical, use factor terms
        cat_offset = len(numeric_cols)
        for i, col in enumerate(categorical_cols):
            formula_terms.append(f(cat_offset + i, lam=lam))
        
        # Combine numeric and categorical data
        X_train_prepared = self._prepare_data(X_train)
        X_val_prepared = self._prepare_data(X_val)
        
        # Fit model
        try:
            gam_terms = formula_terms[0]
            for term in formula_terms[1:]:
                gam_terms += term
            gam = LogisticGAM(terms=gam_terms, max_iter=100)
            gam.gridsearch(X_train_prepared.values, y_train.values, progress=False)
            
            # Predict
            y_score = gam.predict_proba(X_val_prepared.values)
            roc_auc = metrics.roc_auc_score(y_true=y_val, y_score=y_score)
            
            return roc_auc
        except Exception as e:
            print(f"Error in GAM fitting: {e}")
            return 0.5  # Return baseline score on error
    
    def _prepare_data(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Prepare data for GAM
        """
        X_prep = X.copy()
        
        # Encode categorical variables as numeric
        categorical_cols = X_prep.select_dtypes(include=['object', 'category']).columns
        if len(categorical_cols) > 0:
            from sklearn.preprocessing import LabelEncoder
            for col in categorical_cols:
                le = LabelEncoder()
                X_prep[col] = le.fit_transform(X_prep[col].astype(str))
        
        return X_prep
    
    def tuning(self, X_train: pd.DataFrame, y_train: pd.Series, 
               X_val: pd.DataFrame, y_val: pd.Series, n_trials: int) -> dict:
        """
        Hyperparameter tuning using Optuna
        """
        study = optuna.create_study(study_name='GAM', direction='maximize')
        study.optimize(lambda trial: self.objective(trial, X_train, y_train, X_val, y_val), 
                      n_trials=n_trials, show_progress_bar=True)
        best_params = study.best_params
        return best_params
    
    def train(self, X_train: pd.DataFrame, y_train: pd.Series, 
              X_val: pd.DataFrame, y_val: pd.Series, best_params: dict):
        """
        Train final GAM model with best parameters
        """
        n_splines = best_params.get('n_splines', 25)
        lam = best_params.get('lam', 0.6)
        
        # Prepare data
        X_train_prep = self._prepare_data(X_train)
        X_val_prep = self._prepare_data(X_val)
        
        # Build formula
        numeric_cols = X_train_prep.select_dtypes(include=[np.number]).columns
        formula_terms = [s(i, n_splines=n_splines, lam=lam) for i in range(len(numeric_cols))]
        gam_terms = formula_terms[0]
        for term in formula_terms[1:]:
            gam_terms += term
        
        # Fit model
        gam = LogisticGAM(terms=gam_terms, max_iter=100)
        gam.gridsearch(X_train_prep.values, y_train.values, progress=False)
        
        return gam


# Cross-validation function for GAM
def cross_validate_gam_with_oof(X, y, n_splits=5, n_trials=20):
    """
    Cross-validation with OOF predictions for GAM
    """
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)
    
    fold_numbers = np.zeros(len(X))
    oof_predictions_proba = np.zeros(len(X))
    oof_predictions_class = np.zeros(len(X))
    
    fold_results = []
    best_params_per_fold = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'='*70}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'='*70}")
        
        fold_numbers[val_idx] = fold_idx + 1
        
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]
        
        # Split for tuning
        X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
            X_train_fold, y_train_fold, test_size=0.2, random_state=123, stratify=y_train_fold
        )
        
        # Tune
        model_gam = ModelGAM()
        best_params = model_gam.tuning(
            X_train_tune, y_train_tune,
            X_val_tune, y_val_tune,
            n_trials=n_trials
        )
        
        print(f"Best params: {best_params}")
        best_params_per_fold.append(best_params)
        
        # Train
        final_model = model_gam.train(
            X_train_fold, y_train_fold,
            X_val_fold, y_val_fold,
            best_params
        )
        
        # Predict
        X_val_prep = model_gam._prepare_data(X_val_fold)
        y_pred_proba = final_model.predict_proba(X_val_prep.values)
        y_pred_class = (y_pred_proba > 0.5).astype(int)
        
        oof_predictions_proba[val_idx] = y_pred_proba
        oof_predictions_class[val_idx] = y_pred_class
        
        # Metrics
        roc_auc = metrics.roc_auc_score(y_val_fold, y_pred_proba)
        accuracy = metrics.accuracy_score(y_val_fold, y_pred_class)
        
        fold_results.append({
            'fold': fold_idx + 1,
            'roc_auc': roc_auc,
            'accuracy': accuracy,
            'n_train': len(train_idx),
            'n_val': len(val_idx)
        })
        
        print(f"Fold {fold_idx + 1} ROC-AUC: {roc_auc:.4f}")
        print(f"Fold {fold_idx + 1} Accuracy: {accuracy:.4f}")
    
    results_df = pd.DataFrame(fold_results)
    
    print(f"\n{'='*70}")
    print("Cross-Validation Summary")
    print(f"{'='*70}")
    print(f"Mean ROC-AUC: {results_df['roc_auc'].mean():.4f} (+/- {results_df['roc_auc'].std():.4f})")
    print(f"Mean Accuracy: {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")
    
    overall_roc_auc = metrics.roc_auc_score(y, oof_predictions_proba)
    print(f"\nOverall OOF ROC-AUC: {overall_roc_auc:.4f}")
    
    return fold_numbers, oof_predictions_proba, results_df, best_params_per_fold

In [327]:
df_middle_glm_gam = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [328]:
from sklearn.preprocessing import LabelEncoder
for cat in cat_cols:
    le = LabelEncoder()
    le.fit(df_middle_glm_gam[cat])
    df_middle_glm_gam[cat] = le.transform(df_middle_glm_gam[cat])


In [329]:
df_middle_glm_gam['U'] = (df_middle_glm_gam['X_upcoming_premium'] - df_middle_glm_gam['X_policy_premium']) / df_middle_glm_gam['X_policy_premium']

In [330]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


class ModelGLM():
    """GLM replacement using sklearn Logistic Regression."""

    def __init__(self):
        self.categorical_cols = []
        self.numeric_cols = []

    def _build_model(self, C: float) -> Pipeline:
        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])

        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, self.numeric_cols),
                ("cat", categorical_transformer, self.categorical_cols),
            ],
            remainder="drop",
        )

        clf = LogisticRegression(
            C=C,
            max_iter=1000,
            solver="lbfgs",
            random_state=123,
        )

        return Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf),
        ])

    def objective(self, trial, X_train, y_train, X_val, y_val):
        C = trial.suggest_float("C", 1e-3, 100.0, log=True)

        self.categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
        self.numeric_cols = [c for c in X_train.columns if c not in self.categorical_cols]

        model = self._build_model(C=C)
        model.fit(X_train, y_train)
        y_score = model.predict_proba(X_val)[:, 1]
        return metrics.roc_auc_score(y_true=y_val, y_score=y_score)

    def tuning(self, X_train, y_train, X_val, y_val, n_trials):
        study = optuna.create_study(study_name="LogisticRegression", direction="maximize")
        study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=n_trials,
            show_progress_bar=True,
        )
        return study.best_params

    def train(self, X_train, y_train, X_val, y_val, best_params):
        C = best_params.get("C", 1.0)
        self.categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
        self.numeric_cols = [c for c in X_train.columns if c not in self.categorical_cols]

        model = self._build_model(C=C)
        model.fit(X_train, y_train)
        return model

In [331]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn import metrics
import pandas as pd
import numpy as np


class ModelLinearRegressor:
    """Baseline linear regression model."""

    def train(self, X_train: pd.DataFrame, y_train: pd.Series):
        model = LinearRegression()
        model.fit(X_train, y_train)
        return model


def cross_validate_linear_regression(X, y, n_splits=5):
    """
    Run Linear Regression with KFold cross-validation.
    FeatureProcessor is fit on each training fold and applied to both train and val
    to prevent data leakage. U is kept untouched (not processed).
    """
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)

    lin_fold_numbers = np.zeros(len(X))
    lin_oof = np.zeros(len(X))
    lin_results = []

    # Identify columns to keep untouched (U)
    u_cols = [col for col in ["U"] if col in X.columns]
    x_feature_cols = [col for col in X.columns if col not in u_cols]

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
        print(f"\n{'='*70}")
        print(f"Fold {fold_idx}/{n_splits}")
        print(f"{'='*70}")

        X_train_fold = X.iloc[train_idx].reset_index(drop=True)
        X_val_fold = X.iloc[val_idx].reset_index(drop=True)
        y_train_fold = y.iloc[train_idx].reset_index(drop=True)
        y_val_fold = y.iloc[val_idx].reset_index(drop=True)

        lin_fold_numbers[val_idx] = fold_idx

        # --- Separate U from other features ---
        if u_cols:
            U_train = X_train_fold[u_cols].copy()
            U_val = X_val_fold[u_cols].copy()
            X_train_raw = X_train_fold[x_feature_cols].copy()
            X_val_raw = X_val_fold[x_feature_cols].copy()
        else:
            X_train_raw = X_train_fold.copy()
            X_val_raw = X_val_fold.copy()

        # --- Fit preprocessor on train fold only (no leakage) ---
        fp = FeatureProcessor()
        X_train_proc = fp.fit_transform(X_train_raw)
        X_val_proc = fp.transform(X_val_raw)

        # --- Concatenate U back with processed features ---
        if u_cols:
            X_train_proc = pd.concat([U_train.reset_index(drop=True), X_train_proc.reset_index(drop=True)], axis=1)
            X_val_proc = pd.concat([U_val.reset_index(drop=True), X_val_proc.reset_index(drop=True)], axis=1)

        # Linear regression
        linear_model = ModelLinearRegressor().train(X_train_proc, y_train_fold)
        lin_pred = linear_model.predict(X_val_proc)
        lin_oof[val_idx] = lin_pred

        lin_mae = metrics.mean_absolute_error(y_val_fold, lin_pred)
        lin_rmse = np.sqrt(metrics.mean_squared_error(y_val_fold, lin_pred))
        lin_r2 = metrics.r2_score(y_val_fold, lin_pred)

        lin_results.append({
            "fold": fold_idx,
            "model": "LinearRegression",
            "mae": lin_mae,
            "rmse": lin_rmse,
            "r2": lin_r2,
            "n_train": len(train_idx),
            "n_val": len(val_idx),
        })

        print(f"LinearRegression -> MAE: {lin_mae:.4f}, RMSE: {lin_rmse:.4f}, R2: {lin_r2:.4f}")

    lin_df = pd.DataFrame(lin_results)

    print(f"\n{'='*70}")
    print("Overall OOF Metrics")
    print(f"{'='*70}")

    lin_oof_mae = metrics.mean_absolute_error(y, lin_oof)
    lin_oof_rmse = np.sqrt(metrics.mean_squared_error(y, lin_oof))
    lin_oof_r2 = metrics.r2_score(y, lin_oof)

    print(f"LinearRegression OOF -> MAE: {lin_oof_mae:.4f}, RMSE: {lin_oof_rmse:.4f}, R2: {lin_oof_r2:.4f}")

    linear_results = {
        "fold_numbers": lin_fold_numbers,
        "oof_predictions": lin_oof,
        "results_df": lin_df,
        "oof_metrics": {"mae": lin_oof_mae, "rmse": lin_oof_rmse, "r2": lin_oof_r2},
    }

    return linear_results

## Section 15 — Linear Regression: Policy Premium (5-Fold OOF)

### `ModelLinearRegressor`
Plain **sklearn `LinearRegression`** with no hyperparameters — serves as the interpretable baseline for premium prediction.

### `cross_validate_linear_regression`
- 5-fold KFold with no tuning overhead
- Reports MAE, RMSE, R² per fold and overall OOF metrics
- Stores OOF predictions as `linear_reg_oof_prediction` → renamed to `Y_hat`

**Dataset:** `df_middle_reg_linear` — same filtering thresholds as other datasets, with label-encoded categoricals.  
`U = (upcoming_premium − Y) / Y` is computed after column renaming.

**Exported to:** `df_exp_financial_loss_linear_black_box.csv`  
Schema: `id, features, Y (premium), U, Y_hat (predicted premium)`

In [332]:
df_middle_reg_linear = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [333]:
from sklearn.preprocessing import LabelEncoder
for cat in cat_cols:
    le = LabelEncoder()
    le.fit(df_middle_reg_linear[cat])
    df_middle_reg_linear[cat] = le.transform(df_middle_reg_linear[cat])
    

In [334]:
model_features_reg

['X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure']

In [335]:
# Regression setup (Linear Regression only)
X_reg = df_middle_reg_linear.loc[:,model_features_reg]
y_reg = df_middle_reg_linear['X_policy_premium']

linear_results = cross_validate_linear_regression(
    X_reg, y_reg, n_splits=5
)

# Save OOF outputs for downstream analysis
df_middle_reg_linear['linear_reg_oof_prediction'] = linear_results['oof_predictions']


Fold 1/5
LinearRegression -> MAE: 36.4032, RMSE: 51.5802, R2: 0.2525

Fold 2/5
LinearRegression -> MAE: 36.0190, RMSE: 50.7388, R2: 0.2541

Fold 3/5
LinearRegression -> MAE: 36.0655, RMSE: 50.9690, R2: 0.2527

Fold 4/5
LinearRegression -> MAE: 36.0176, RMSE: 50.6448, R2: 0.2541

Fold 5/5
LinearRegression -> MAE: 36.0272, RMSE: 50.6479, R2: 0.2555

Overall OOF Metrics
LinearRegression OOF -> MAE: 36.1065, RMSE: 50.9174, R2: 0.2538


In [336]:
df_middle_reg_linear.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'linear_reg_oof_prediction'],
      dtype='str')

In [337]:
df_middle_reg_linear.columns = ['id','X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'Y', 'X_prev_renewal_perc',
       'X_upcoming_premium', 'X_cur_renewal_perc', 'X_year', '1-Z','Y_hat']

In [338]:
df_middle_reg_linear['U'] = (df_middle_reg_linear['X_upcoming_premium'] - df_middle_reg_linear['Y']) / df_middle_reg_linear['Y']

In [339]:
df_middle_reg_linear['U+1'] = 1 + df_middle_reg_linear['U']

In [345]:
df_middle_reg_linear.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'Y', 'X_prev_renewal_perc',
       'X_upcoming_premium', 'X_cur_renewal_perc', 'X_year', '1-Z', 'Y_hat',
       'U', 'U+1'],
      dtype='str')

In [346]:
df_middle_reg_linear.loc[:,['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                     'Y','U','U+1','Y_hat']].to_csv('df_exp_financial_loss_linear_black_box_feat_preprocessor.csv',sep=';')

## Section 16 — Model Serialisation

Trained models are persisted as **pickle files** in the `artifacts/` directory for reuse without retraining.

### Saved models

| File | Model | Task |
|---|---|---|
| `artifacts/xgb_classifier_churn.pkl` | XGBoost classifier | Churn probability |
| `artifacts/xgb_regressor_policy_premium.pkl` | XGBoost regressor | Policy premium |
| `artifacts/glm_logistic_churn.pkl` | Logistic Regression (GLM) | Churn probability |
| `artifacts/linear_regression_policy_premium.pkl` | Linear Regression | Policy premium |

Models are loaded and retrained only if not already present in the kernel namespace (`globals()` check), enabling fast re-runs of downstream cells without full retraining.

> **Note:** Feature order must be preserved when loading models for inference.  
> Use `model.feature_names_in_` (available on sklearn estimators) to validate column alignment.

In [350]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split


ARTIFACTS_DIR = Path("artifacts_preproc_pipeline")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CLS_PICKLE_PATH = ARTIFACTS_DIR / "xgb_classifier_no_preprocessor.pkl"
REG_PICKLE_PATH = ARTIFACTS_DIR / "xgb_regressor_no_preprocessor.pkl"


def _is_xgb_fitted(model) -> bool:
    try:
        _ = model.get_booster()
        return True
    except Exception:
        return False


# ---------------------------
# 1) Build / retrieve classifier
# ---------------------------
if "xgb_classifier_model" in globals() and isinstance(globals()["xgb_classifier_model"], xgb.XGBClassifier):
    xgb_classifier_model = globals()["xgb_classifier_model"]
else:
    X_cls = df_middle_class.loc[:, model_features_class].copy()
    y_cls = df_middle_class["1-Z"].astype(int).copy()

    Xc_train, Xc_val, yc_train, yc_val = train_test_split(
        X_cls,
        y_cls,
        test_size=0.2,
        random_state=123,
        stratify=y_cls,
    )

    cls_params = {
        "random_state": 123,
        "n_estimators": 250,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "eval_metric": "logloss",
        "enable_categorical": True,
        "early_stopping_rounds": 20,
    }
    xgb_classifier_model = xgb.XGBClassifier(**cls_params)
    xgb_classifier_model.fit(Xc_train, yc_train, eval_set=[(Xc_val, yc_val)], verbose=0)


# ---------------------------
# 2) Build / retrieve regressor
# ---------------------------
if "xgb_model" in globals() and isinstance(globals()["xgb_model"], xgb.XGBRegressor):
    xgb_regressor_model = globals()["xgb_model"]
else:
    xgb_regressor_model = None

# If the existing regressor is not fitted, fit one now.
if xgb_regressor_model is None or not _is_xgb_fitted(xgb_regressor_model):
    X_reg = df_middle.loc[:, model_features_reg].copy()
    y_reg_local = df_middle["Y"].copy()

    Xr_train, Xr_val, yr_train, yr_val = train_test_split(
        X_reg,
        y_reg_local,
        test_size=0.2,
        random_state=123,
    )

    reg_params = {
        "random_state": 123,
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "objective": "reg:squarederror",
        "enable_categorical": True,
        "early_stopping_rounds": 20,
    }
    xgb_regressor_model = xgb.XGBRegressor(**reg_params)
    xgb_regressor_model.fit(Xr_train, yr_train, eval_set=[(Xr_val, yr_val)], verbose=0)


# ---------------------------
# 3) Save to pickle
# ---------------------------
with open(CLS_PICKLE_PATH, "wb") as f:
    pickle.dump(xgb_classifier_model, f)

with open(REG_PICKLE_PATH, "wb") as f:
    pickle.dump(xgb_regressor_model, f)

print(f"Saved classifier to: {CLS_PICKLE_PATH.resolve()}")
print(f"Saved regressor to:  {REG_PICKLE_PATH.resolve()}")

Saved classifier to: C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts_preproc_pipeline\xgb_classifier_no_preprocessor.pkl
Saved regressor to:  C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts_preproc_pipeline\xgb_regressor_no_preprocessor.pkl


In [351]:
model_features_class

['X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure',
 'X_policy_premium',
 'U']

In [349]:
model_features_reg

['X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure']

## Section 17 — Single-Record Inference (Smoke Test)

Loads the serialised XGBoost classifier and regressor from `artifacts/` and runs a single-record prediction to verify the pickle round-trip is correct.

**Features:**
- **FeatureProcessor with PCA Whitening**: Each model pipeline includes preprocessing that applies multivariate whitening
- **Mahalanobis Distance**: Whitening ensures Euclidean distance in transformed space = Mahalanobis distance in original space
- **U Column Handling**: The premium change ratio `U` is kept untouched (not preprocessed)
- **Dimensionality Reduction**: Components are selected to explain 95% of variance
- **Complete Pipeline**: Each pickle contains `{preprocessor, model, u_cols, x_feature_cols}` for reproducible inference

**Checks performed:**
- Classifier: outputs churn probability and hard class label
- Regressor: outputs predicted policy premium
- Both: preprocessing is applied consistently using saved FeatureProcessor
- Displays both original and preprocessed (PCA-whitened) features

This cell is intended as a **quick sanity check** before handing off models to production or the optimisation module.

## Section 18 — Save All Four Models with FeatureProcessor (PCA Whitening)

Final cell: trains and serialises all four models with integrated **FeatureProcessor** featuring **PCA whitening**:

1. **XGBClassifier** — churn (on `df_middle_class`) + PCA preprocessing
2. **XGBRegressor** — premium (on `df_middle`) + PCA preprocessing
3. **GLM logistic** — churn (on `df_middle_glm`) + PCA preprocessing
4. **LinearRegression** — premium (on `df_middle_reg_linear`) + PCA preprocessing

**Preprocessing pipeline (`FeatureProcessor` with `use_pca=True`):**
- **Whitening**: Transforms data so covariance matrix = Identity
- **Dimensionality reduction**: Retains components explaining 95% of variance
- **U preservation**: Premium change ratio `U` is excluded from processing and preserved in original scale
- **Variance reporting**: Displays number of components kept and variance explained for each model

Each model is saved as a **complete pipeline** containing:
- `preprocessor`: fitted FeatureProcessor with PCA whitening
- `model`: trained sklearn/xgboost model
- `u_cols`: list of untouched columns (e.g., ['U'])
- `x_feature_cols`: list of feature columns to preprocess

**Benefits:**
- **Consistent preprocessing**: Same transformations applied in training and inference
- **No data leakage**: Preprocessor fitted only on training data
- **Mahalanobis distance**: Whitening makes distance metrics more meaningful
- **Production-ready**: Single pickle file contains everything needed for inference

In [356]:
import pickle
from pathlib import Path

import pandas as pd


ARTIFACTS_DIR = Path("artifacts_preproc_pipeline")
CLS_PICKLE_PATH = ARTIFACTS_DIR / "xgb_classifier_churn_feat_preproc.pkl"
REG_PICKLE_PATH = ARTIFACTS_DIR / "xgb_regressor_policy_premium_feat_preproc.pkl"


# 1) Load model pipelines from pickle
with open(CLS_PICKLE_PATH, "rb") as f:
    cls_pipeline = pickle.load(f)

with open(REG_PICKLE_PATH, "rb") as f:
    reg_pipeline = pickle.load(f)

print("Loaded XGBClassifier pipeline components:")
print(f"  - Preprocessor: {type(cls_pipeline['preprocessor']).__name__}")
print(f"  - Model: {type(cls_pipeline['model']).__name__}")
print(f"  - U columns: {cls_pipeline['u_cols']}")
print(f"  - Feature columns: {len(cls_pipeline['x_feature_cols'])} features\n")

print("Loaded XGBRegressor pipeline components:")
print(f"  - Preprocessor: {type(reg_pipeline['preprocessor']).__name__}")
print(f"  - Model: {type(reg_pipeline['model']).__name__}")
print(f"  - U columns: {reg_pipeline['u_cols']}")
print(f"  - Feature columns: {len(reg_pipeline['x_feature_cols'])} features\n")


# 2) Build a single-record input for each model
single_cls_record = df_middle_class.loc[:, model_features_class].iloc[[0]].copy()
single_reg_record = df_middle.loc[:, model_features_reg].iloc[[0]].copy()


# 3) Preprocess inputs using the saved preprocessors
def preprocess_for_inference(X, pipeline):
    """Apply preprocessing pipeline for inference"""
    u_cols = pipeline['u_cols']
    x_feature_cols = pipeline['x_feature_cols']
    preprocessor = pipeline['preprocessor']
    
    if u_cols:
        U_data = X[u_cols].copy()
        X_features = X[x_feature_cols].copy()
    else:
        X_features = X.copy()
    
    # Transform features
    X_proc = preprocessor.transform(X_features)
    
    # Concatenate U back if present
    if u_cols:
        X_preprocessed = pd.concat([U_data.reset_index(drop=True), X_proc.reset_index(drop=True)], axis=1)
    else:
        X_preprocessed = X_proc
    
    return X_preprocessed


# Preprocess classifier input
single_cls_preprocessed = preprocess_for_inference(single_cls_record, cls_pipeline)

# Preprocess regressor input
single_reg_preprocessed = preprocess_for_inference(single_reg_record, reg_pipeline)


# 4) Inference on preprocessed data
cls_prob = float(cls_pipeline['model'].predict_proba(single_cls_preprocessed)[:, 1][0])
cls_pred = int(cls_pipeline['model'].predict(single_cls_preprocessed)[0])
reg_pred = float(reg_pipeline['model'].predict(single_reg_preprocessed)[0])

print("="*70)
print("Single-record classifier inference (with PCA whitening preprocessing):")
print("="*70)
print(f"  Predicted churn probability: {cls_prob:.6f}")
print(f"  Predicted churn class:       {cls_pred}")

print("\n" + "="*70)
print("Single-record regressor inference (with PCA whitening preprocessing):")
print("="*70)
print(f"  Predicted premium:           {reg_pred:.6f}")

print("\n" + "="*70)
print("Input record used for classifier (original):")
print("="*70)
print(single_cls_record)

print("\n" + "="*70)
print("Input record used for classifier (preprocessed with PCA):")
print("="*70)
print(single_cls_preprocessed)

print("\n" + "="*70)
print("Input record used for regressor (original):")
print("="*70)
print(single_reg_record)

print("\n" + "="*70)
print("Input record used for regressor (preprocessed with PCA):")
print("="*70)
print(single_reg_preprocessed)

Loaded XGBClassifier pipeline components:
  - Preprocessor: FeatureProcessor
  - Model: XGBClassifier
  - U columns: ['U']
  - Feature columns: 10 features

Loaded XGBRegressor pipeline components:
  - Preprocessor: FeatureProcessor
  - Model: XGBRegressor
  - U columns: []
  - Feature columns: 9 features

Single-record classifier inference (with PCA whitening preprocessing):
  Predicted churn probability: 0.232544
  Predicted churn class:       0

Single-record regressor inference (with PCA whitening preprocessing):
  Predicted premium:           200.003983

Input record used for classifier (original):
   X_age  X_bonus_malus_rating  X_distr_channel  X_vehicle_type  X_ttm_claims  \
7   93.0                  45.0                5               7             0   

   X_policy_count  X_risk_code  X_vehicle_age  X_policy_tenure  \
7               1            0           24.0             24.0   

   X_policy_premium         U  
7          175.7725  0.123099  

Input record used for classi

In [242]:
df_middle_reg.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'U'],
      dtype='str')

In [357]:
# Save all requested models as pickle WITH FeatureProcessor (PCA whitening):
# 1) XGBClassifier (churn) + preprocessor
# 2) XGBRegressor (premium) + preprocessor
# 3) GLM logistic model (churn) + preprocessor
# 4) Linear model (premium) + preprocessor

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


ARTIFACTS_DIR = Path("artifacts_preproc_pipeline")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PATH_XGB_CLASSIFIER = ARTIFACTS_DIR / "xgb_classifier_churn_feat_preproc.pkl"
PATH_XGB_REGRESSOR = ARTIFACTS_DIR / "xgb_regressor_policy_premium_feat_preproc.pkl"
PATH_GLM_LOGISTIC = ARTIFACTS_DIR / "glm_logistic_churn_feat_preproc.pkl"
PATH_LINEAR_MODEL = ARTIFACTS_DIR / "linear_regression_policy_premium_feat_preproc.pkl"


def _to_xgb_compatible(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    obj_cols = df2.select_dtypes(include=["object"]).columns
    for col in obj_cols:
        df2[col] = df2[col].astype("category")
    return df2


def _is_fitted_xgb(model) -> bool:
    try:
        _ = model.get_booster()
        return True
    except Exception:
        return False


def preprocess_with_U_separation(X, use_pca=True, n_components=None, explained_variance_threshold=0.95):
    """
    Preprocess features with FeatureProcessor, keeping U untouched.
    
    Returns:
    --------
    X_preprocessed : DataFrame with U (if present) + processed features
    preprocessor : fitted FeatureProcessor
    """
    # Identify U column
    u_cols = [col for col in ["U"] if col in X.columns]
    x_feature_cols = [col for col in X.columns if col not in u_cols]
    
    if u_cols:
        U_data = X[u_cols].copy()
        X_raw = X[x_feature_cols].copy()
    else:
        X_raw = X.copy()
    
    # Fit FeatureProcessor with PCA whitening
    preprocessor = FeatureProcessor(
        use_pca=use_pca,
        n_components=n_components,
        explained_variance_threshold=explained_variance_threshold
    )
    X_proc = preprocessor.fit_transform(X_raw)
    
    # Concatenate U back
    if u_cols:
        X_preprocessed = pd.concat([U_data.reset_index(drop=True), X_proc.reset_index(drop=True)], axis=1)
    else:
        X_preprocessed = X_proc
    
    return X_preprocessed, preprocessor


# ----------------------------
# XGB Classifier (churn)
# ----------------------------
print("="*70)
print("Training XGBClassifier with FeatureProcessor (PCA whitening)...")
print("="*70)

X_cls = df_middle_class.loc[:, model_features_class].copy()
y_cls = df_middle_class["1-Z"].astype(int).copy()

# Split first
Xc_train_raw, Xc_val_raw, yc_train, yc_val = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=123, stratify=y_cls
)

# Preprocess with U separation and PCA whitening
Xc_train_proc, cls_preprocessor = preprocess_with_U_separation(
    Xc_train_raw, 
    use_pca=True, 
    explained_variance_threshold=0.95
)

# Transform validation set with fitted preprocessor
u_cols_cls = [col for col in ["U"] if col in Xc_val_raw.columns]
x_feature_cols_cls = [col for col in Xc_val_raw.columns if col not in u_cols_cls]

if u_cols_cls:
    U_val = Xc_val_raw[u_cols_cls].copy()
    Xc_val_raw_features = Xc_val_raw[x_feature_cols_cls].copy()
else:
    Xc_val_raw_features = Xc_val_raw.copy()

Xc_val_proc = cls_preprocessor.transform(Xc_val_raw_features)

if u_cols_cls:
    Xc_val_proc = pd.concat([U_val.reset_index(drop=True), Xc_val_proc.reset_index(drop=True)], axis=1)

# Convert to XGB compatible format
Xc_train_proc = _to_xgb_compatible(Xc_train_proc)
Xc_val_proc = _to_xgb_compatible(Xc_val_proc)

# Train XGBClassifier
xgb_classifier_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=123,
    enable_categorical=True,
    early_stopping_rounds=20,
)
xgb_classifier_model.fit(Xc_train_proc, yc_train, eval_set=[(Xc_val_proc, yc_val)], verbose=0)

# Get variance info
var_info_cls = cls_preprocessor.get_explained_variance_info()
print(f"Classifier - Components kept: {var_info_cls['n_components_kept']}")
print(f"Classifier - Variance explained: {var_info_cls['cumulative_variance_ratio'][var_info_cls['n_components_kept']-1]:.4f}")


# ----------------------------
# XGB Regressor (policy_premium)
# ----------------------------
print("\n" + "="*70)
print("Training XGBRegressor with FeatureProcessor (PCA whitening)...")
print("="*70)

X_reg = df_middle.loc[:, model_features_reg].copy()
y_reg_local = df_middle["Y"].copy()

# Split first
Xr_train_raw, Xr_val_raw, yr_train, yr_val = train_test_split(
    X_reg, y_reg_local, test_size=0.2, random_state=123
)

# Preprocess with PCA whitening (no U for regressor)
Xr_train_proc, reg_preprocessor = preprocess_with_U_separation(
    Xr_train_raw, 
    use_pca=True, 
    explained_variance_threshold=0.95
)

# Transform validation set
Xr_val_proc = reg_preprocessor.transform(Xr_val_raw)

# Convert to XGB compatible format
Xr_train_proc = _to_xgb_compatible(Xr_train_proc)
Xr_val_proc = _to_xgb_compatible(Xr_val_proc)

# Train XGBRegressor
xgb_regressor_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=123,
    enable_categorical=True,
    early_stopping_rounds=20,
)
xgb_regressor_model.fit(Xr_train_proc, yr_train, eval_set=[(Xr_val_proc, yr_val)], verbose=0)

# Get variance info
var_info_reg = reg_preprocessor.get_explained_variance_info()
print(f"Regressor - Components kept: {var_info_reg['n_components_kept']}")
print(f"Regressor - Variance explained: {var_info_reg['cumulative_variance_ratio'][var_info_reg['n_components_kept']-1]:.4f}")


# ----------------------------
# GLM logistic model (churn)
# ----------------------------
print("\n" + "="*70)
print("Training GLM Logistic with FeatureProcessor (PCA whitening)...")
print("="*70)

X_glm = df_middle_glm.loc[:, model_features_class].copy()
y_glm = df_middle_glm["1-Z"].astype(int).copy()

# Split first
Xg_train_raw, Xg_val_raw, yg_train, yg_val = train_test_split(
    X_glm, y_glm, test_size=0.2, random_state=123, stratify=y_glm
)

# Preprocess with U separation and PCA whitening
Xg_train_proc, glm_preprocessor = preprocess_with_U_separation(
    Xg_train_raw, 
    use_pca=True, 
    explained_variance_threshold=0.95
)

# Transform validation set
u_cols_glm = [col for col in ["U"] if col in Xg_val_raw.columns]
x_feature_cols_glm = [col for col in Xg_val_raw.columns if col not in u_cols_glm]

if u_cols_glm:
    U_val_glm = Xg_val_raw[u_cols_glm].copy()
    Xg_val_raw_features = Xg_val_raw[x_feature_cols_glm].copy()
else:
    Xg_val_raw_features = Xg_val_raw.copy()

Xg_val_proc = glm_preprocessor.transform(Xg_val_raw_features)

if u_cols_glm:
    Xg_val_proc = pd.concat([U_val_glm.reset_index(drop=True), Xg_val_proc.reset_index(drop=True)], axis=1)

# Train GLM
glm_model_obj = ModelGLM()
glm_logistic_model = glm_model_obj.train(
    Xg_train_proc,
    yg_train,
    Xg_val_proc,
    yg_val,
    {"C": 1.0},
)

var_info_glm = glm_preprocessor.get_explained_variance_info()
print(f"GLM - Components kept: {var_info_glm['n_components_kept']}")
print(f"GLM - Variance explained: {var_info_glm['cumulative_variance_ratio'][var_info_glm['n_components_kept']-1]:.4f}")


# ----------------------------
# Linear model (policy_premium)
# ----------------------------
print("\n" + "="*70)
print("Training Linear Regression with FeatureProcessor (PCA whitening)...")
print("="*70)

if "df_middle_reg_linear" in globals():
    X_lin = df_middle_reg_linear.loc[:, model_features_reg].copy()
    y_lin = df_middle_reg_linear["Y"].copy()
else:
    X_lin = df_middle.loc[:, model_features_reg].copy()
    y_lin = df_middle["Y"].copy()

# Preprocess with PCA whitening (no U for regressor)
X_lin_proc, linear_preprocessor = preprocess_with_U_separation(
    X_lin, 
    use_pca=True, 
    explained_variance_threshold=0.95
)

# Train Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_lin_proc, y_lin)

var_info_lin = linear_preprocessor.get_explained_variance_info()
print(f"Linear - Components kept: {var_info_lin['n_components_kept']}")
print(f"Linear - Variance explained: {var_info_lin['cumulative_variance_ratio'][var_info_lin['n_components_kept']-1]:.4f}")


# ----------------------------
# Save all models WITH PREPROCESSING
# ----------------------------
print("\n" + "="*70)
print("Saving models with preprocessors...")
print("="*70)

# Save as dictionary with both model and preprocessor
xgb_cls_pipeline = {
    "preprocessor": cls_preprocessor,
    "model": xgb_classifier_model,
    "u_cols": u_cols_cls,
    "x_feature_cols": x_feature_cols_cls,
}

xgb_reg_pipeline = {
    "preprocessor": reg_preprocessor,
    "model": xgb_regressor_model,
    "u_cols": [],  # No U for regressor
    "x_feature_cols": model_features_reg,
}

glm_pipeline = {
    "preprocessor": glm_preprocessor,
    "model": glm_logistic_model,
    "u_cols": u_cols_glm,
    "x_feature_cols": x_feature_cols_glm,
}

linear_pipeline = {
    "preprocessor": linear_preprocessor,
    "model": linear_model,
    "u_cols": [],  # No U for regressor
    "x_feature_cols": model_features_reg,
}

with open(PATH_XGB_CLASSIFIER, "wb") as f:
    pickle.dump(xgb_cls_pipeline, f)

with open(PATH_XGB_REGRESSOR, "wb") as f:
    pickle.dump(xgb_reg_pipeline, f)

with open(PATH_GLM_LOGISTIC, "wb") as f:
    pickle.dump(glm_pipeline, f)

with open(PATH_LINEAR_MODEL, "wb") as f:
    pickle.dump(linear_pipeline, f)

print(f"\n✓ Saved: {PATH_XGB_CLASSIFIER.resolve()}")
print(f"✓ Saved: {PATH_XGB_REGRESSOR.resolve()}")
print(f"✓ Saved: {PATH_GLM_LOGISTIC.resolve()}")
print(f"✓ Saved: {PATH_LINEAR_MODEL.resolve()}")
print("\nAll models saved with FeatureProcessor (PCA whitening)!")
print("Each pickle contains: {'preprocessor', 'model', 'u_cols', 'x_feature_cols'}")

Training XGBClassifier with FeatureProcessor (PCA whitening)...
Classifier - Components kept: 2
Classifier - Variance explained: 0.9545

Training XGBRegressor with FeatureProcessor (PCA whitening)...
Regressor - Components kept: 4
Regressor - Variance explained: 0.9626

Training GLM Logistic with FeatureProcessor (PCA whitening)...
GLM - Components kept: 2
GLM - Variance explained: 0.9545

Training Linear Regression with FeatureProcessor (PCA whitening)...
Linear - Components kept: 4
Linear - Variance explained: 0.9628

Saving models with preprocessors...

✓ Saved: C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts_preproc_pipeline\xgb_classifier_churn_feat_preproc.pkl
✓ Saved: C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts_preproc_pipeline\xgb_regressor_policy_premium_feat_preproc.pkl
✓ Saved: C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts_preproc_pipeline\glm_logi